# Mortgage Data Quality & Reporting Pipeline
**Author:** Rehan Ahmed  
**Tools:** Python, pandas, NumPy, SQLite, SQL and OpenPyXL  
**Scope:** 2024 HMDA public records for Miami-Dade County, Florida (county code `12086`).

## Project introduction
This portfolio project demonstrates how to turn public mortgage records into traceable analytical tables and management reports. It focuses on data collection, profiling, validation, exception management, relational storage, SQL reporting and documentation—the practical responsibilities of a junior data analyst or data-quality analyst.

**Business question:** How can an analyst produce reliable mortgage activity reports while retaining ambiguous records and making data-quality concerns visible?

The pipeline preserves raw values, separates parsing failures from review warnings, tests its rules, reconciles record counts and exports a formatted Excel report. It does not make lending decisions and is not affiliated with Pegasus Mortgage Lending.

## Dataset introduction
The input `county_12086.csv` contains **82,691 records and 99 columns** covering **737 lender identifiers**. Every row is a reported loan/application record, not necessarily a unique borrower, a completed loan, or an active application. Outcomes include originations, denials, withdrawals, incomplete files, purchased loans and preapproval outcomes.

The download interface identified the release as the **2024 One Year dataset**. Its SHA-256 fingerprint is recorded programmatically to identify this exact download.

- [HMDA dataset browser](https://ffiec.cfpb.gov/data-browser/data/)
- [Public field definitions](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields/)
- [2024 data publications](https://ffiec.cfpb.gov/data-publication/2024)

Public data is privacy-modified. Some values are grouped or rounded, and direct application identifiers and credit scores are not available. `applicant_credit_score_type` describes a scoring model, not the actual score. Blank, `NA` and `Exempt` must remain distinguishable. The exported `loan_to_value_ratio` column is retained under its source name; it should not be assumed to equal public loan amount divided by public property value.

## How to run
1. Open this notebook in Google Colab and upload `county_12086.csv` to `/content`.
2. Run the code cells in order. Markdown cells provide explanations and require no execution.
3. If required, install `pandas`, `numpy`, `openpyxl` and `jinja2` in your environment. SQLite and the other standard-library imports need no separate installation.
4. Keep the input unchanged. Outputs are written under `/content/mortgage_project/`.
5. Download the workbook, backup ZIP and notebook before the Colab runtime expires.

**Execution note:** Code and saved outputs are preserved from the supplied notebook. This edit adds documentation only; saved outputs describe the existing run. The workflow still uses Colab-specific paths and downloads. On another platform, adapt those paths and skip the `google.colab` download calls.

## Workflow
Raw CSV → profiling → preparation → validation and review flags → isolated tests → SQLite snapshot → reconciled SQL reports → Excel and backup.

## Result snapshot
| Measure | Verified result |
|---|---:|
| Input records | 82,691 |
| Rules implemented | 16 (11 ERROR, 5 WARNING) |
| Records passing without warnings | 81,147 |
| Records with review warnings | 1,544 (1.87%) |
| Warning events | 1,646 |
| Quarantined records | 0 |
| Isolated validation cases | 22 passed |
| Additional scenarios | Duplicate-group and overlapping-warning tests passed |
| Originated loans | 33,869 |
| Lenders with originations | 610 |

These counts apply to the supplied source fingerprint. Zero ERROR findings means no failures of the implemented rules, not universally error-free data.


## 1. Source ingestion and profiling

### Cell 01 — Import libraries and configure display

**Purpose and method:** Load file-path, hashing and tabular-analysis tools. Display settings make wide tables easier to inspect; they do not modify data.

**Expected result / interpretation:** A setup confirmation.


In [1]:
from pathlib import Path
import hashlib
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

print("Setup complete.")

Setup complete.


### Cell 02 — Locate the source CSV

**Purpose and method:** Check the expected Colab path and fail clearly if the file is absent. File size helps confirm the intended upload.

**Expected result / interpretation:** The filename and approximately 29.50 MiB (labelled MB by the code).


In [2]:
DATA_PATH = Path("/content/county_12086.csv")

if not DATA_PATH.exists():
    available_files = [p.name for p in Path("/content").glob("*.csv")]
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.name}. "
        f"CSV files available: {available_files}"
    )

print("File:", DATA_PATH.name)
print(f"Size: {DATA_PATH.stat().st_size / (1024 ** 2):.2f} MB")

File: county_12086.csv
Size: 29.50 MB


### Cell 03 — Load values without implicit missing-value conversion

**Purpose and method:** Read every column as text and disable automatic NA conversion. This preserves geography codes and separates literal NA, Exempt and empty fields.

**Expected result / interpretation:** 82,691 rows, 99 columns and a five-row preview.


In [3]:
raw_df = pd.read_csv(
    DATA_PATH,
    dtype="string",
    keep_default_na=False,
    na_filter=False,
    encoding="utf-8-sig"
)

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]:,}")

display(raw_df.head(5))

Rows: 82,691
Columns: 99


,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,derived_race,derived_sex,action_taken,purchaser_type,preapproval,...,aus-2,aus-3,aus-4,aus-5,denial_reason-1,denial_reason-2,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2024,549300LYRWPSYPK6S325,33124,FL,12086,12086010806,C,FHA:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,Race Not Available,Sex Not Available,6,2,2,...,,,,,10,,,,3989,93.33,79400,56,556,1110,23
1,2024,549300LYRWPSYPK6S325,33124,FL,12086,12086011409,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,Race Not Available,Sex Not Available,6,2,2,...,,,,,10,,,,5642,78.23,79400,96,582,1100,24
2,2024,549300LYRWPSYPK6S325,33124,FL,12086,12086009803,C,FHA:First Lien,Single Family (1-4 Units):Site-Built,Hispanic or Latino,White,Joint,1,2,2,...,,,,,10,,,,7564,90.98,79400,103,1259,1235,39
3,2024,549300LYRWPSYPK6S325,33124,FL,12086,12086010806,C,FHA:First Lien,Single Family (1-4 Units):Site-Built,Hispanic or Latino,Race Not Available,Male,2,0,2,...,,,,,10,,,,3989,93.33,79400,56,556,1110,23
4,2024,549300LYRWPSYPK6S325,33124,FL,12086,12086012701,C,FHA:First Lien,Single Family (1-4 Units):Site-Built,Hispanic or Latino,White,Joint,4,0,2,...,3,,,,10,,,,10996,91.64,79400,157,2115,2456,6


### Cell 04 — Fingerprint the input

**Purpose and method:** Hash the complete source file in chunks and capture its size and shape. The fingerprint identifies the exact bytes used for this run, not a borrower.

**Expected result / interpretation:** A source metadata table with a SHA-256 value.


In [4]:
hasher = hashlib.sha256()

with DATA_PATH.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        hasher.update(chunk)

source_metadata = {
    "filename": DATA_PATH.name,
    "file_size_bytes": DATA_PATH.stat().st_size,
    "sha256": hasher.hexdigest(),
    "row_count": len(raw_df),
    "column_count": len(raw_df.columns),
}

display(
    pd.DataFrame(
        source_metadata.items(),
        columns=["Property", "Value"]
    )
)

,Property,Value
0,filename,county_12086.csv
1,file_size_bytes,30928203
2,sha256,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...
3,row_count,82691
4,column_count,99


### Cell 05 — Inventory the columns

**Purpose and method:** List field names, positions and loaded types. String types at this stage are deliberate; conversion happens in separate derived columns.

**Expected result / interpretation:** A 99-field schema inventory.


In [5]:
column_inventory = pd.DataFrame({
    "position": range(1, len(raw_df.columns) + 1),
    "column_name": raw_df.columns,
    "loaded_dtype": [str(dtype) for dtype in raw_df.dtypes]
})

display(column_inventory)

,position,column_name,loaded_dtype
0,1,activity_year,string
1,2,lei,string
2,3,derived_msa-md,string
3,4,state_code,string
4,5,county_code,string
...,...,...,...
94,95,ffiec_msa_md_median_family_income,string
95,96,tract_to_msa_income_percentage,string
96,97,tract_owner_occupied_units,string
97,98,tract_one_to_four_family_homes,string


### Cell 06 — Check required fields

**Purpose and method:** Verify the initial 15 core fields before downstream operations. This list is a core check, not a complete dependency manifest for every later report.

**Expected result / interpretation:** All 15 required fields present.


In [6]:
required_columns = [
    "activity_year",
    "lei",
    "state_code",
    "county_code",
    "action_taken",
    "loan_type",
    "loan_purpose",
    "lien_status",
    "loan_amount",
    "loan_to_value_ratio",
    "interest_rate",
    "loan_term",
    "property_value",
    "income",
    "debt_to_income_ratio",
]

missing_columns = [
    column for column in required_columns
    if column not in raw_df.columns
]

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print(f"PASS: All {len(required_columns)} required columns are present.")

PASS: All 15 required columns are present.


### Cell 07 — Verify dataset scope

**Purpose and method:** Count records matching the chosen year and county, inspect state values and count lender LEIs. Scope rules apply to this project download.

**Expected result / interpretation:** All rows match 2024/12086; six state values are NA; 737 lender identifiers.


In [7]:
scope_checks = pd.DataFrame([
    {
        "check": "Reporting year is 2024",
        "matching_rows": int(raw_df["activity_year"].eq("2024").sum()),
        "other_rows": int(raw_df["activity_year"].ne("2024").sum()),
    },
    {
        "check": "County code is 12086",
        "matching_rows": int(raw_df["county_code"].eq("12086").sum()),
        "other_rows": int(raw_df["county_code"].ne("12086").sum()),
    },
])

display(scope_checks)

print("State values:")
display(
    raw_df["state_code"]
    .value_counts(dropna=False)
    .rename_axis("state_code")
    .reset_index(name="records")
)

print(f"Distinct lender identifiers: {raw_df['lei'].nunique():,}")

,check,matching_rows,other_rows
0,Reporting year is 2024,82691,0
1,County code is 12086,82691,0


State values:


,state_code,records
0,FL,82685
1,NA,6


Distinct lender identifiers: 737


### Cell 08 — Profile missing and special representations

**Purpose and method:** Count blanks, NA, Exempt, distinct values and surrounding whitespace separately. Optional secondary fields may legitimately be blank.

**Expected result / interpretation:** A field profile; percentages may round to 100% even when a few values exist.


In [8]:
profile_rows = []

for column in raw_df.columns:
    original = raw_df[column]
    stripped = original.str.strip()

    profile_rows.append({
        "column": column,
        "distinct_values": original.nunique(dropna=False),
        "blank_count": int(stripped.eq("").sum()),
        "NA_count": int(stripped.eq("NA").sum()),
        "Exempt_count": int(stripped.eq("Exempt").sum()),
        "surrounding_whitespace_count": int(original.ne(stripped).sum()),
    })

field_profile = pd.DataFrame(profile_rows)

field_profile["blank_pct"] = (
    field_profile["blank_count"] / len(raw_df) * 100
).round(2)

display(
    field_profile.sort_values(
        ["blank_count", "NA_count"],
        ascending=False
    ).head(25)
)

,column,distinct_values,blank_count,NA_count,Exempt_count,surrounding_whitespace_count,blank_pct
70,co-applicant_race-5,2,82690,0,0,0,100.00
69,co-applicant_race-4,3,82689,0,0,0,100.00
57,co-applicant_ethnicity-4,2,82687,0,0,0,100.00
58,co-applicant_ethnicity-5,2,82687,0,0,0,100.00
53,applicant_ethnicity-5,2,82683,0,0,0,99.99
65,applicant_race-5,5,82683,0,0,0,99.99
52,applicant_ethnicity-4,3,82679,0,0,0,99.99
64,applicant_race-4,10,82670,0,0,0,99.97
68,co-applicant_race-3,9,82666,0,0,0,99.97
56,co-applicant_ethnicity-3,6,82581,0,0,0,99.87


### Cell 09 — Summarize recorded outcomes

**Purpose and method:** Map action codes to readable labels and reconcile category counts to the input. Shares use all records, including purchased loans and preapprovals.

**Expected result / interpretation:** Eight outcome categories totalling 82,691 records; not a credit-decision approval rate.


In [9]:
action_labels = {
    "1": "Loan originated",
    "2": "Application approved but not accepted",
    "3": "Application denied",
    "4": "Application withdrawn",
    "5": "File closed for incompleteness",
    "6": "Purchased loan",
    "7": "Preapproval request denied",
    "8": "Preapproval request approved but not accepted",
}

outcome_summary = (
    raw_df["action_taken"]
    .value_counts(dropna=False)
    .rename_axis("action_code")
    .reset_index(name="records")
)

outcome_summary["outcome"] = (
    outcome_summary["action_code"]
    .map(action_labels)
    .fillna("Unrecognized code — review required")
)

outcome_summary["share_of_all_records_pct"] = (
    outcome_summary["records"] / len(raw_df) * 100
).round(2)

display(
    outcome_summary[
        ["action_code", "outcome", "records", "share_of_all_records_pct"]
    ]
)

assert outcome_summary["records"].sum() == len(raw_df)
print("PASS: Outcome counts reconcile to the input row count.")

,action_code,outcome,records,share_of_all_records_pct
0,1,Loan originated,33869,40.96
1,3,Application denied,20868,25.24
2,4,Application withdrawn,11709,14.16
3,6,Purchased loan,7593,9.18
4,5,File closed for incompleteness,5663,6.85
5,2,Application approved but not accepted,2908,3.52
6,8,Preapproval request approved but not accepted,63,0.08
7,7,Preapproval request denied,18,0.02


PASS: Outcome counts reconcile to the input row count.


### Cell 10 — Inspect DTI representations

**Purpose and method:** Count exact numeric strings, ranges and special values before deciding how to parse them. Never replace a reported range with an invented exact value.

**Expected result / interpretation:** The observed DTI categories and frequencies.


In [10]:
dti_summary = (
    raw_df["debt_to_income_ratio"]
    .value_counts(dropna=False)
    .rename_axis("raw_dti_value")
    .reset_index(name="records")
)

display(dti_summary)

,raw_dti_value,records
0,NA,31339
1,>60%,6869
2,50%-60%,6505
3,30%-<36%,4734
4,20%-<30%,4050
5,49,3756
6,<20%,3241
7,44,2267
8,48,2205
9,42,2043


### Cell 11 — Profile candidate numeric fields

**Purpose and method:** Temporarily parse six fields to inspect magnitudes and unexpected text without replacing raw data. Raw income units differ from loan amounts, so do not compare the magnitudes directly.

**Expected result / interpretation:** Numeric/special counts, extrema and medians; zero unexpected-text findings in this file.


In [11]:
numeric_fields = [
    "loan_amount",
    "loan_to_value_ratio",
    "interest_rate",
    "loan_term",
    "property_value",
    "income",
]

numeric_profile_rows = []

for column in numeric_fields:
    values = raw_df[column].str.strip()
    numeric_values = pd.to_numeric(values, errors="coerce")

    recognized_special = values.isin(["", "NA", "Exempt"])
    unexpected_text = numeric_values.isna() & ~recognized_special

    numeric_profile_rows.append({
        "column": column,
        "numeric_count": int(numeric_values.notna().sum()),
        "special_or_blank_count": int(recognized_special.sum()),
        "unexpected_text_count": int(unexpected_text.sum()),
        "negative_count": int(numeric_values.lt(0).sum()),
        "zero_count": int(numeric_values.eq(0).sum()),
        "minimum": numeric_values.min(),
        "median": numeric_values.median(),
        "maximum": numeric_values.max(),
    })

numeric_profile = pd.DataFrame(numeric_profile_rows)

display(numeric_profile)

,column,numeric_count,special_or_blank_count,unexpected_text_count,negative_count,zero_count,minimum,median,maximum
0,loan_amount,82691,0,0,0,0,5000.000,315000.000,8.440550e+08
1,loan_to_value_ratio,54657,28034,0,0,0,0.003,72.900,1.764706e+04
2,interest_rate,43993,38698,0,0,1124,0.000,6.999,1.687500e+01
3,loan_term,80641,2050,0,0,0,1.000,360.000,6.420000e+02
4,property_value,62633,20058,0,0,0,5000.000,575000.000,1.500005e+09
5,income,69000,13691,0,101,1735,-11051.000,131.000,1.400000e+05


### Cell 12 — Inspect exact public-row matches

**Purpose and method:** Compare all 99 source fields. Count additional copies and all members of matching groups separately. Public-data transformations can make distinct records identical.

**Expected result / interpretation:** 219 occurrences beyond the first; 415 group members. No rows removed.


In [12]:
repeat_after_first = raw_df.duplicated(keep="first")
all_repeat_members = raw_df.duplicated(keep=False)

print(
    "Exact repeated rows beyond the first occurrence:",
    f"{repeat_after_first.sum():,}"
)

print(
    "Total rows belonging to exact-match groups:",
    f"{all_repeat_members.sum():,}"
)

preview_columns = [
    "activity_year",
    "lei",
    "county_code",
    "action_taken",
    "loan_amount",
    "income",
    "interest_rate",
]

display(
    raw_df.loc[all_repeat_members, preview_columns].head(10)
)

Exact repeated rows beyond the first occurrence: 219
Total rows belonging to exact-match groups: 415


,activity_year,lei,county_code,action_taken,loan_amount,income,interest_rate
72,2024,549300LYRWPSYPK6S325,12086,4,355000.0,NA,NA
135,2024,549300LYRWPSYPK6S325,12086,4,355000.0,NA,NA
462,2024,549300LYRWPSYPK6S325,12086,5,515000.0,NA,NA
3670,2024,549300JKYBEV1DF8FD32,12086,4,105000.0,NA,Exempt
3674,2024,549300JKYBEV1DF8FD32,12086,4,105000.0,NA,Exempt
3759,2024,254900OGMHGN3FQRRB74,12086,1,855000.0,NA,Exempt
3785,2024,254900OGMHGN3FQRRB74,12086,1,925000.0,NA,Exempt
3815,2024,254900OGMHGN3FQRRB74,12086,1,925000.0,NA,Exempt
3818,2024,254900OGMHGN3FQRRB74,12086,1,855000.0,NA,Exempt
4643,2024,SS1TRMSN6BRNMOREEV51,12086,4,2505000.0,1219,NA


## 2. Preparation and source preservation

### Cell 13 — Create the traceable working copy

**Purpose and method:** Add the file fingerprint and a one-based source record number. Together they uniquely locate a record within this file; they are not persistent loan identifiers.

**Expected result / interpretation:** 82,691 preserved records with tracking fields.


In [13]:
prepared_df = raw_df.copy(deep=True)

prepared_df.insert(
    0,
    "source_record_number",
    range(1, len(prepared_df) + 1)
)

prepared_df.insert(
    0,
    "source_file_sha256",
    source_metadata["sha256"]
)

assert prepared_df["source_record_number"].is_unique
assert len(prepared_df) == len(raw_df)

print(f"Working copy created: {len(prepared_df):,} records.")
print("Original columns and values preserved.")

Working copy created: 82,691 records.
Original columns and values preserved.


### Cell 14 — Parse numeric values with statuses

**Purpose and method:** Convert finite numbers to nullable Float64 values. Companion statuses retain NUMERIC, BLANK, NA, EXEMPT or INVALID so an analytical null does not erase source meaning.

**Expected result / interpretation:** Six pairs of derived value/status columns. Numeric syntax does not establish business validity.


In [14]:
import numpy as np

def parse_numeric_field(series):
    text = series.str.strip()

    parsed = pd.to_numeric(text, errors="coerce").astype("Float64")

    is_finite = pd.Series(
        np.isfinite(
            parsed.to_numpy(dtype=float, na_value=np.nan)
        ),
        index=series.index
    )

    status = pd.Series(
        "INVALID",
        index=series.index,
        dtype="string"
    )

    status.loc[is_finite] = "NUMERIC"
    status.loc[text.eq("")] = "BLANK"
    status.loc[text.eq("NA")] = "NA"
    status.loc[text.eq("Exempt")] = "EXEMPT"

    numeric = parsed.where(status.eq("NUMERIC"), pd.NA)

    return numeric, status


for column in numeric_fields:
    numeric, status = parse_numeric_field(prepared_df[column])

    prepared_df[f"{column}_numeric"] = numeric
    prepared_df[f"{column}_parse_status"] = status

print("Numeric values and parse statuses created.")

display(
    prepared_df[
        [
            "income",
            "income_numeric",
            "income_parse_status",
            "interest_rate",
            "interest_rate_numeric",
            "interest_rate_parse_status",
        ]
    ].head(10)
)

Numeric values and parse statuses created.


,income,income_numeric,income_parse_status,interest_rate,interest_rate_numeric,interest_rate_parse_status
0,NA,<NA>,NA,6.500,6.5,NUMERIC
1,NA,<NA>,NA,6.500,6.5,NUMERIC
2,101,101.0,NUMERIC,5.875,5.875,NUMERIC
3,NA,<NA>,NA,6.990,6.99,NUMERIC
4,196,196.0,NUMERIC,NA,<NA>,NA
5,NA,<NA>,NA,5.990,5.99,NUMERIC
6,NA,<NA>,NA,6.990,6.99,NUMERIC
7,118,118.0,NUMERIC,7.375,7.375,NUMERIC
8,269,269.0,NUMERIC,NA,<NA>,NA
9,155,155.0,NUMERIC,5.990,5.99,NUMERIC


### Cell 15 — Reconcile numeric parsing

**Purpose and method:** Count each parse category and assert that each field accounts for every input row.

**Expected result / interpretation:** All six totals equal 82,691; INVALID is zero.


In [15]:
parse_categories = ["NUMERIC", "BLANK", "NA", "EXEMPT", "INVALID"]

parse_summary = pd.DataFrame({
    column: (
        prepared_df[f"{column}_parse_status"]
        .value_counts()
        .reindex(parse_categories, fill_value=0)
    )
    for column in numeric_fields
}).T

parse_summary.index.name = "field"
parse_summary["TOTAL"] = parse_summary[parse_categories].sum(axis=1)

assert parse_summary["TOTAL"].eq(len(raw_df)).all()

display(parse_summary)

print("PASS: Every numeric field reconciles to the input record count.")

,NUMERIC,BLANK,NA,EXEMPT,INVALID,TOTAL
field,,,,,,
loan_amount,82691,0,0,0,0,82691
loan_to_value_ratio,54657,0,27512,522,0,82691
interest_rate,43993,0,38176,522,0,82691
loan_term,80641,0,1528,522,0,82691
property_value,62633,0,19536,522,0,82691
income,69000,0,13691,0,0,82691


PASS: Every numeric field reconciles to the input record count.


### Cell 16 — Separate exact DTI from ranges

**Purpose and method:** Store exact numeric DTI only when reported as a number. Preserve range labels and special representations independently.

**Expected result / interpretation:** 25,431 exact values, 25,399 ranges, 31,339 NA and 522 Exempt.


In [16]:
dti_text = prepared_df["debt_to_income_ratio"].str.strip()

known_dti_ranges = {
    "<20%",
    "20%-<30%",
    "30%-<36%",
    "50%-60%",
    ">60%",
}

dti_numeric, dti_base_status = parse_numeric_field(
    prepared_df["debt_to_income_ratio"]
)

dti_status = dti_base_status.replace({"NUMERIC": "EXACT_NUMERIC"})
dti_status.loc[dti_text.isin(known_dti_ranges)] = "REPORTED_RANGE"

prepared_df["dti_representation"] = dti_status

prepared_df["dti_exact_pct"] = dti_numeric.where(
    dti_status.eq("EXACT_NUMERIC"),
    pd.NA
)

prepared_df["dti_reported_range"] = dti_text.where(
    dti_status.eq("REPORTED_RANGE"),
    pd.NA
)

dti_parse_summary = (
    prepared_df["dti_representation"]
    .value_counts()
    .rename_axis("representation")
    .reset_index(name="records")
)

assert dti_parse_summary["records"].sum() == len(raw_df)

display(dti_parse_summary)

,representation,records
0,NA,31339
1,EXACT_NUMERIC,25431
2,REPORTED_RANGE,25399
3,EXEMPT,522


### Cell 17 — Add outcome labels and review flags

**Purpose and method:** Flag unavailable state, full-row matches, negative income and zero interest without rejecting records. A record can have multiple warnings.

**Expected result / interpretation:** 1,544 distinct flagged records; individual flag totals overlap.


In [17]:
prepared_df["action_label"] = (
    prepared_df["action_taken"]
    .map(action_labels)
    .fillna("UNRECOGNIZED_CODE")
)

prepared_df["review_state_unavailable"] = (
    prepared_df["state_code"].str.strip().isin(["", "NA"])
)

prepared_df["review_exact_row_match"] = (
    raw_df.duplicated(keep=False)
)

prepared_df["review_negative_income"] = (
    prepared_df["income_numeric"].lt(0).fillna(False)
)

prepared_df["review_zero_interest_rate"] = (
    prepared_df["interest_rate_numeric"].eq(0).fillna(False)
)

review_columns = [
    "review_state_unavailable",
    "review_exact_row_match",
    "review_negative_income",
    "review_zero_interest_rate",
]

review_summary = pd.DataFrame({
    "review_flag": review_columns,
    "flagged_records": [
        int(prepared_df[column].sum())
        for column in review_columns
    ],
})

display(review_summary)

print(
    "Distinct records with at least one review flag:",
    f"{prepared_df[review_columns].any(axis=1).sum():,}"
)

print("Flags can overlap; their counts must not be added as unique records.")

,review_flag,flagged_records
0,review_state_unavailable,6
1,review_exact_row_match,415
2,review_negative_income,101
3,review_zero_interest_rate,1124


Distinct records with at least one review flag: 1,544
Flags can overlap; their counts must not be added as unique records.


### Cell 18 — Verify source preservation

**Purpose and method:** Compare all original columns and types with the raw frame and verify tracking-key uniqueness.

**Expected result / interpretation:** Unchanged source values; 99 original columns and 121 prepared columns.


In [18]:
pd.testing.assert_frame_equal(
    prepared_df[raw_df.columns],
    raw_df,
    check_dtype=True
)

assert len(prepared_df) == source_metadata["row_count"]

assert prepared_df[
    ["source_file_sha256", "source_record_number"]
].duplicated().sum() == 0

print("PASS: All original source values and data types are unchanged.")
print("PASS: No records were removed or added.")
print("PASS: Source tracking keys are unique.")

print(f"\nOriginal columns: {len(raw_df.columns)}")
print(f"Prepared columns: {len(prepared_df.columns)}")

PASS: All original source values and data types are unchanged.
PASS: No records were removed or added.
PASS: Source tracking keys are unique.

Original columns: 99
Prepared columns: 121


## 3. Validation and exception routing

### Cell 19 — Document the validation rules

**Purpose and method:** Register rule identifiers, severity, affected fields and explanations. ERROR rules block reporting in this project; WARNING rules retain records for review.

**Expected result / interpretation:** 16 rules. These are selected technical checks, not comprehensive regulatory validation.


In [19]:
rule_definitions = [
    (
        "ERR_YEAR_SCOPE", "ERROR", "activity_year",
        "Reporting year must match this project's 2024 scope."
    ),
    (
        "ERR_COUNTY_SCOPE", "ERROR", "county_code",
        "County must match this project's 12086 scope."
    ),
    (
        "ERR_LEI_FORMAT", "ERROR", "lei",
        "Lender identifier must contain 20 uppercase letters or digits; "
        "this does not verify registration or checksum."
    ),
    (
        "ERR_ACTION_CODE", "ERROR", "action_taken",
        "Action code must be one of the documented codes 1 through 8."
    ),
    (
        "ERR_LOAN_AMOUNT", "ERROR", "loan_amount",
        "Loan amount must parse as a finite number greater than zero."
    ),
]

# Check numeric syntax separately from business meaning.
for field in numeric_fields:
    if field != "loan_amount":
        rule_definitions.append((
            f"ERR_PARSE_{field.upper()}",
            "ERROR",
            field,
            "Unexpected numeric text or a non-finite value. "
            "Blank, NA, and Exempt are tracked separately."
        ))

rule_definitions.extend([
    (
        "ERR_DTI_REPRESENTATION", "ERROR", "debt_to_income_ratio",
        "DTI must use a recognized numeric, range, or special representation."
    ),
    (
        "WARN_STATE_UNAVAILABLE", "WARNING", "state_code",
        "State is blank or NA; review against the county."
    ),
    (
        "WARN_STATE_SCOPE", "WARNING", "state_code",
        "A supplied state value differs from FL for this county scope."
    ),
    (
        "WARN_EXACT_ROW_MATCH", "WARNING", "__entire_source_row__",
        "Entire source row matches another row; retain pending review."
    ),
    (
        "WARN_NEGATIVE_INCOME", "WARNING", "income",
        "Reported income is negative; investigate without changing it."
    ),
    (
        "WARN_ZERO_INTEREST", "WARNING", "interest_rate",
        "Reported interest rate is zero; investigate without changing it."
    ),
])

rule_register = pd.DataFrame(
    rule_definitions,
    columns=["rule_id", "severity", "field", "description"]
)

assert rule_register["rule_id"].is_unique

display(rule_register)

,rule_id,severity,field,description
0,ERR_YEAR_SCOPE,ERROR,activity_year,Reporting year must match this project's 2024 scope.
1,ERR_COUNTY_SCOPE,ERROR,county_code,County must match this project's 12086 scope.
2,ERR_LEI_FORMAT,ERROR,lei,Lender identifier must contain 20 uppercase letters or d...
3,ERR_ACTION_CODE,ERROR,action_taken,Action code must be one of the documented codes 1 throug...
4,ERR_LOAN_AMOUNT,ERROR,loan_amount,Loan amount must parse as a finite number greater than z...
5,ERR_PARSE_LOAN_TO_VALUE_RATIO,ERROR,loan_to_value_ratio,"Unexpected numeric text or a non-finite value. Blank, NA..."
6,ERR_PARSE_INTEREST_RATE,ERROR,interest_rate,"Unexpected numeric text or a non-finite value. Blank, NA..."
7,ERR_PARSE_LOAN_TERM,ERROR,loan_term,"Unexpected numeric text or a non-finite value. Blank, NA..."
8,ERR_PARSE_PROPERTY_VALUE,ERROR,property_value,"Unexpected numeric text or a non-finite value. Blank, NA..."
9,ERR_PARSE_INCOME,ERROR,income,"Unexpected numeric text or a non-finite value. Blank, NA..."


### Cell 20 — Evaluate rules into an exception table

**Purpose and method:** Create one event for each record-rule match and retain the raw value plus explanation. Masks are evaluated on prepared values; preparation must run again after changing raw inputs.

**Expected result / interpretation:** 1,646 warning events with record-level traceability.


In [20]:
def evaluate_rules(frame, register):
    state = frame["state_code"].str.strip()

    masks = {
        "ERR_YEAR_SCOPE":
            frame["activity_year"].ne("2024"),

        "ERR_COUNTY_SCOPE":
            frame["county_code"].ne("12086"),

        "ERR_LEI_FORMAT":
            ~frame["lei"].str.fullmatch(r"[A-Z0-9]{20}", na=False),

        "ERR_ACTION_CODE":
            ~frame["action_taken"].isin(action_labels.keys()),

        "ERR_LOAN_AMOUNT":
            frame["loan_amount_parse_status"].ne("NUMERIC")
            | frame["loan_amount_numeric"].le(0).fillna(False),

        "ERR_DTI_REPRESENTATION":
            frame["dti_representation"].eq("INVALID"),

        "WARN_STATE_UNAVAILABLE":
            state.isin(["", "NA"]),

        "WARN_STATE_SCOPE":
            ~state.isin(["FL", "", "NA"]),

        "WARN_EXACT_ROW_MATCH":
            frame["review_exact_row_match"],

        "WARN_NEGATIVE_INCOME":
            frame["income_numeric"].lt(0).fillna(False),

        "WARN_ZERO_INTEREST":
            frame["interest_rate_numeric"].eq(0).fillna(False),
    }

    for field in numeric_fields:
        if field != "loan_amount":
            masks[f"ERR_PARSE_{field.upper()}"] = (
                frame[f"{field}_parse_status"].eq("INVALID")
            )

    assert set(masks) == set(register["rule_id"])

    exception_columns = [
        "source_file_sha256",
        "source_record_number",
        "rule_id",
        "severity",
        "field",
        "raw_value",
        "description",
    ]

    parts = []

    for rule in register.itertuples(index=False):
        mask = masks[rule.rule_id].fillna(False).astype(bool)

        if not mask.any():
            continue

        part = frame.loc[
            mask,
            ["source_file_sha256", "source_record_number"]
        ].copy()

        part["rule_id"] = rule.rule_id
        part["severity"] = rule.severity
        part["field"] = rule.field

        if rule.field == "__entire_source_row__":
            part["raw_value"] = "[Exact match across all original columns]"
        else:
            part["raw_value"] = frame.loc[mask, rule.field].to_numpy()

        part["description"] = rule.description
        parts.append(part[exception_columns])

    if not parts:
        return pd.DataFrame(columns=exception_columns)

    return (
        pd.concat(parts, ignore_index=True)
        .sort_values(["source_record_number", "rule_id"])
        .reset_index(drop=True)
    )


exceptions_df = evaluate_rules(prepared_df, rule_register)

print(f"Exception events: {len(exceptions_df):,}")
display(exceptions_df.head(10))

Exception events: 1,646


,source_file_sha256,source_record_number,rule_id,severity,field,raw_value,description
0,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,73,WARN_EXACT_ROW_MATCH,WARNING,__entire_source_row__,[Exact match across all original columns],Entire source row matches another row; retain pending re...
1,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,136,WARN_EXACT_ROW_MATCH,WARNING,__entire_source_row__,[Exact match across all original columns],Entire source row matches another row; retain pending re...
2,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,463,WARN_EXACT_ROW_MATCH,WARNING,__entire_source_row__,[Exact match across all original columns],Entire source row matches another row; retain pending re...
3,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,891,WARN_ZERO_INTEREST,WARNING,interest_rate,0,Reported interest rate is zero; investigate without chan...
4,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,925,WARN_ZERO_INTEREST,WARNING,interest_rate,0,Reported interest rate is zero; investigate without chan...
5,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,1037,WARN_ZERO_INTEREST,WARNING,interest_rate,0,Reported interest rate is zero; investigate without chan...
6,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,1038,WARN_ZERO_INTEREST,WARNING,interest_rate,0,Reported interest rate is zero; investigate without chan...
7,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,1070,WARN_ZERO_INTEREST,WARNING,interest_rate,0,Reported interest rate is zero; investigate without chan...
8,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,1268,WARN_NEGATIVE_INCOME,WARNING,income,-26,Reported income is negative; investigate without changin...
9,63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29b...,1315,WARN_NEGATIVE_INCOME,WARNING,income,-456,Reported income is negative; investigate without changin...


### Cell 21 — Summarize every rule

**Purpose and method:** Left-join event counts to the rule register so rules with zero findings remain visible.

**Expected result / interpretation:** Zero ERROR events and 1,646 WARNING events.


In [21]:
rule_counts = (
    exceptions_df.groupby("rule_id")
    .size()
    .rename("exception_events")
    .reset_index()
)

validation_summary = rule_register.merge(
    rule_counts,
    on="rule_id",
    how="left",
    validate="one_to_one"
)

validation_summary["exception_events"] = (
    validation_summary["exception_events"]
    .fillna(0)
    .astype(int)
)

display(
    validation_summary[
        ["rule_id", "severity", "field", "exception_events"]
    ]
)

print(
    "ERROR events:",
    int(exceptions_df["severity"].eq("ERROR").sum())
)

print(
    "WARNING events:",
    int(exceptions_df["severity"].eq("WARNING").sum())
)

,rule_id,severity,field,exception_events
0,ERR_YEAR_SCOPE,ERROR,activity_year,0
1,ERR_COUNTY_SCOPE,ERROR,county_code,0
2,ERR_LEI_FORMAT,ERROR,lei,0
3,ERR_ACTION_CODE,ERROR,action_taken,0
4,ERR_LOAN_AMOUNT,ERROR,loan_amount,0
5,ERR_PARSE_LOAN_TO_VALUE_RATIO,ERROR,loan_to_value_ratio,0
6,ERR_PARSE_INTEREST_RATE,ERROR,interest_rate,0
7,ERR_PARSE_LOAN_TERM,ERROR,loan_term,0
8,ERR_PARSE_PROPERTY_VALUE,ERROR,property_value,0
9,ERR_PARSE_INCOME,ERROR,income,0


ERROR events: 0
WARNING events: 1646


### Cell 22 — Assign record status

**Purpose and method:** Assign PASS, REVIEW or ERROR, with ERROR taking precedence. This separates distinct record counts from event counts.

**Expected result / interpretation:** 81,147 PASS; 1,544 REVIEW; zero ERROR.


In [22]:
error_ids = set(
    exceptions_df.loc[
        exceptions_df["severity"].eq("ERROR"),
        "source_record_number"
    ]
)

warning_ids = set(
    exceptions_df.loc[
        exceptions_df["severity"].eq("WARNING"),
        "source_record_number"
    ]
)

prepared_df["validation_status"] = pd.Series(
    "PASS",
    index=prepared_df.index,
    dtype="string"
)

prepared_df.loc[
    prepared_df["source_record_number"].isin(warning_ids),
    "validation_status"
] = "REVIEW"

# ERROR takes priority if a record has both errors and warnings.
prepared_df.loc[
    prepared_df["source_record_number"].isin(error_ids),
    "validation_status"
] = "ERROR"

status_summary = (
    prepared_df["validation_status"]
    .value_counts()
    .reindex(["PASS", "REVIEW", "ERROR"], fill_value=0)
    .rename_axis("validation_status")
    .reset_index(name="records")
)

display(status_summary)

assert status_summary["records"].sum() == len(raw_df)

print("PASS: Validation status counts reconcile to the source.")

,validation_status,records
0,PASS,81147
1,REVIEW,1544
2,ERROR,0


PASS: Validation status counts reconcile to the source.


### Cell 23 — Route records without deletion

**Purpose and method:** Create reporting and quarantine subsets while retaining the full prepared table. Review warnings remain in reporting.

**Expected result / interpretation:** 82,691 reporting-eligible records and zero quarantined records.


In [23]:
reporting_df = prepared_df.loc[
    prepared_df["validation_status"].ne("ERROR")
].copy()

quarantine_df = prepared_df.loc[
    prepared_df["validation_status"].eq("ERROR")
].copy()

assert len(reporting_df) + len(quarantine_df) == len(raw_df)

assert set(reporting_df["source_record_number"]).isdisjoint(
    set(quarantine_df["source_record_number"])
)

pd.testing.assert_frame_equal(
    prepared_df[raw_df.columns],
    raw_df,
    check_dtype=True
)

print(f"Reporting-eligible records: {len(reporting_df):,}")
print(f"Quarantined records: {len(quarantine_df):,}")
print("PASS: All input records accounted for; source values unchanged.")

Reporting-eligible records: 82,691
Quarantined records: 0
PASS: All input records accounted for; source values unchanged.


## 4. Reusable transformations and tests

### Cell 24 — Make preparation reusable

**Purpose and method:** Package transformations into a function so altered test inputs receive fresh numeric values and flags rather than stale derived values.

**Expected result / interpretation:** A reusable preparation function matching the earlier transformation steps.


In [24]:
def prepare_records(source_df, source_fingerprint):
    frame = source_df.copy(deep=True)

    frame.insert(
        0, "source_record_number",
        range(1, len(frame) + 1)
    )
    frame.insert(
        0, "source_file_sha256",
        source_fingerprint
    )

    # Recalculate numeric values and their statuses.
    for field in numeric_fields:
        numeric, status = parse_numeric_field(frame[field])
        frame[f"{field}_numeric"] = numeric
        frame[f"{field}_parse_status"] = status

    # Recalculate DTI representation.
    dti_text = frame["debt_to_income_ratio"].str.strip()
    dti_numeric, dti_status = parse_numeric_field(
        frame["debt_to_income_ratio"]
    )

    dti_status = dti_status.replace({
        "NUMERIC": "EXACT_NUMERIC"
    })

    dti_status.loc[
        dti_text.isin(known_dti_ranges)
    ] = "REPORTED_RANGE"

    frame["dti_representation"] = dti_status

    frame["dti_exact_pct"] = dti_numeric.where(
        dti_status.eq("EXACT_NUMERIC"), pd.NA
    )

    frame["dti_reported_range"] = dti_text.where(
        dti_status.eq("REPORTED_RANGE"), pd.NA
    )

    frame["action_label"] = (
        frame["action_taken"]
        .map(action_labels)
        .fillna("UNRECOGNIZED_CODE")
    )

    # Recalculate review flags using this input batch.
    frame["review_state_unavailable"] = (
        frame["state_code"].str.strip().isin(["", "NA"])
    )

    frame["review_exact_row_match"] = (
        source_df.duplicated(keep=False)
    )

    frame["review_negative_income"] = (
        frame["income_numeric"].lt(0).fillna(False)
    )

    frame["review_zero_interest_rate"] = (
        frame["interest_rate_numeric"].eq(0).fillna(False)
    )

    return frame


print("Reusable preparation function ready.")

Reusable preparation function ready.


### Cell 25 — Check refactoring equivalence

**Purpose and method:** Rebuild preparation and exceptions, then compare both with the earlier results. This verifies that refactoring did not change behavior.

**Expected result / interpretation:** Both frame-equality assertions pass.


In [25]:
rebuilt_df = prepare_records(
    raw_df,
    source_metadata["sha256"]
)

pd.testing.assert_frame_equal(
    rebuilt_df,
    prepared_df.drop(columns=["validation_status"])
)

rebuilt_exceptions = evaluate_rules(
    rebuilt_df,
    rule_register
)

pd.testing.assert_frame_equal(
    rebuilt_exceptions,
    exceptions_df
)

print("PASS: Reusable preparation reproduces existing values.")
print("PASS: Validation reproduces the existing exception table.")

del rebuilt_df, rebuilt_exceptions

PASS: Reusable preparation reproduces existing values.
PASS: Validation reproduces the existing exception table.


### Cell 26 — Define isolated test cases

**Purpose and method:** Take a passing source record as a baseline and make controlled copies with invalid or boundary inputs. Expected rule sets are explicit. These copies are test fixtures, not source observations.

**Expected result / interpretation:** 22 cases covering supported parsing, scope checks and warnings.


In [26]:
baseline_index = prepared_df.index[
    prepared_df["validation_status"].eq("PASS")
][0]

baseline_record = raw_df.loc[baseline_index].copy()

# Each case: name, source-field changes, expected rule IDs.
test_cases = [
    (
        "Unchanged valid record",
        {},
        set()
    ),
    (
        "Wrong reporting year",
        {"activity_year": "2023"},
        {"ERR_YEAR_SCOPE"}
    ),
    (
        "Wrong county",
        {"county_code": "99999"},
        {"ERR_COUNTY_SCOPE"}
    ),
    (
        "Malformed lender identifier",
        {"lei": "BAD_ID"},
        {"ERR_LEI_FORMAT"}
    ),
    (
        "Unknown action code",
        {"action_taken": "99"},
        {"ERR_ACTION_CODE"}
    ),
    (
        "Zero loan amount",
        {"loan_amount": "0"},
        {"ERR_LOAN_AMOUNT"}
    ),
    (
        "Missing loan amount",
        {"loan_amount": ""},
        {"ERR_LOAN_AMOUNT"}
    ),
    (
        "Nonnumeric loan amount",
        {"loan_amount": "not_a_number"},
        {"ERR_LOAN_AMOUNT"}
    ),
    (
        "Malformed DTI",
        {"debt_to_income_ratio": "unknown"},
        {"ERR_DTI_REPRESENTATION"}
    ),
    (
        "Recognized DTI range",
        {"debt_to_income_ratio": "30%-<36%"},
        set()
    ),
    (
        "Unavailable state",
        {"state_code": "NA"},
        {"WARN_STATE_UNAVAILABLE"}
    ),
    (
        "Different state",
        {"state_code": "CA"},
        {"WARN_STATE_SCOPE"}
    ),
    (
        "Negative income",
        {"income": "-10"},
        {"WARN_NEGATIVE_INCOME"}
    ),
    (
        "Zero interest rate",
        {"interest_rate": "0"},
        {"WARN_ZERO_INTEREST"}
    ),
]

# Exercise every remaining numeric parsing rule.
for field in numeric_fields:
    if field != "loan_amount":
        test_cases.append((
            f"Unexpected text in {field}",
            {field: "not_a_number"},
            {f"ERR_PARSE_{field.upper()}"}
        ))

test_cases.extend([
    (
        "Infinite interest rate",
        {"interest_rate": "inf"},
        {"ERR_PARSE_INTEREST_RATE"}
    ),
    (
        "NA interest rate preserved",
        {"interest_rate": "NA"},
        set()
    ),
    (
        "Exempt interest rate preserved",
        {"interest_rate": "Exempt"},
        set()
    ),
])

print(f"Defined {len(test_cases)} isolated test cases.")

Defined 22 isolated test cases.


### Cell 27 — Execute exact-rule tests

**Purpose and method:** Compare expected and actual rule sets, detecting missed findings and unexpected extra flags. The tests cover implemented behavior, not all conceivable business conditions.

**Expected result / interpretation:** All 22 cases pass.


In [27]:
test_results = []

for name, changes, expected_rules in test_cases:
    test_record = baseline_record.copy()

    for field, value in changes.items():
        test_record[field] = value

    test_raw = (
        test_record.to_frame()
        .T
        .reset_index(drop=True)
        .astype("string")
    )

    test_prepared = prepare_records(
        test_raw,
        "TEST_ONLY_NOT_A_SOURCE_FILE_HASH"
    )

    test_exceptions = evaluate_rules(
        test_prepared,
        rule_register
    )

    actual_rules = set(test_exceptions["rule_id"])

    test_results.append({
        "test": name,
        "expected_rules": ", ".join(sorted(expected_rules)) or "None",
        "actual_rules": ", ".join(sorted(actual_rules)) or "None",
        "passed": actual_rules == expected_rules,
    })

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

assert test_results_df["passed"].all(), (
    "At least one validation test failed. "
    "Inspect the rows where passed is False."
)

print(f"PASS: All {len(test_results_df)} isolated tests passed.")

,test,expected_rules,actual_rules,passed
0,Unchanged valid record,None,None,True
1,Wrong reporting year,ERR_YEAR_SCOPE,ERR_YEAR_SCOPE,True
2,Wrong county,ERR_COUNTY_SCOPE,ERR_COUNTY_SCOPE,True
3,Malformed lender identifier,ERR_LEI_FORMAT,ERR_LEI_FORMAT,True
4,Unknown action code,ERR_ACTION_CODE,ERR_ACTION_CODE,True
5,Zero loan amount,ERR_LOAN_AMOUNT,ERR_LOAN_AMOUNT,True
6,Missing loan amount,ERR_LOAN_AMOUNT,ERR_LOAN_AMOUNT,True
7,Nonnumeric loan amount,ERR_LOAN_AMOUNT,ERR_LOAN_AMOUNT,True
8,Malformed DTI,ERR_DTI_REPRESENTATION,ERR_DTI_REPRESENTATION,True
9,Recognized DTI range,None,None,True


PASS: All 22 isolated tests passed.


### Cell 28 — Test matching rows and overlapping events

**Purpose and method:** Verify that both members of a duplicate group receive warnings and that one record can generate two events. Confirm tests remain separate from real data.

**Expected result / interpretation:** Three PASS messages for duplicate handling, overlap and source isolation.


In [28]:
# Test A: Two identical records must BOTH receive the match warning.
duplicate_test_raw = pd.DataFrame([
    baseline_record.to_dict(),
    baseline_record.to_dict(),
]).astype("string")

duplicate_test_prepared = prepare_records(
    duplicate_test_raw,
    "TEST_ONLY_DUPLICATES"
)

duplicate_test_exceptions = evaluate_rules(
    duplicate_test_prepared,
    rule_register
)

assert set(duplicate_test_exceptions["rule_id"]) == {
    "WARN_EXACT_ROW_MATCH"
}
assert len(duplicate_test_exceptions) == 2
assert duplicate_test_exceptions["source_record_number"].nunique() == 2

print("PASS: Both identical test records receive a review warning.")


# Test B: One record can generate two separate warning events.
overlap_record = baseline_record.copy()
overlap_record["income"] = "-10"
overlap_record["interest_rate"] = "0"

overlap_raw = (
    overlap_record.to_frame()
    .T
    .reset_index(drop=True)
    .astype("string")
)

overlap_prepared = prepare_records(
    overlap_raw,
    "TEST_ONLY_OVERLAPPING_WARNINGS"
)

overlap_exceptions = evaluate_rules(
    overlap_prepared,
    rule_register
)

assert set(overlap_exceptions["rule_id"]) == {
    "WARN_NEGATIVE_INCOME",
    "WARN_ZERO_INTEREST",
}
assert len(overlap_exceptions) == 2
assert overlap_exceptions["source_record_number"].nunique() == 1

print("PASS: Two warning events correctly refer to one test record.")


# Confirm that test data has not entered the real dataset.
assert prepared_df["source_file_sha256"].eq(
    source_metadata["sha256"]
).all()

pd.testing.assert_frame_equal(
    prepared_df[raw_df.columns],
    raw_df
)

print("PASS: Real source data remains unchanged and separate from tests.")

PASS: Both identical test records receive a review warning.
PASS: Two warning events correctly refer to one test record.
PASS: Real source data remains unchanged and separate from tests.


## 5. Relational storage and reconciliation

### Cell 29 — Set up SQLite inputs

**Purpose and method:** Create a dedicated snapshot location and check keys, row counts and rule references before loading.

**Expected result / interpretation:** A database path and successful pre-load checks.


In [29]:
import sqlite3

DB_DIRECTORY = Path("/content/mortgage_project/db")
DB_DIRECTORY.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIRECTORY / "mortgage_qa_snapshot.db"

# Confirm that only the real source is being loaded.
assert prepared_df["source_file_sha256"].eq(
    source_metadata["sha256"]
).all()

assert len(prepared_df) == source_metadata["row_count"]

assert not prepared_df[
    ["source_file_sha256", "source_record_number"]
].duplicated().any()

assert not exceptions_df[
    ["source_file_sha256", "source_record_number", "rule_id"]
].duplicated().any()

assert set(exceptions_df["rule_id"]).issubset(
    set(rule_register["rule_id"])
)

print("Database destination:", DB_PATH)
print("PASS: Database inputs checked.")

Database destination: /content/mortgage_project/db/mortgage_qa_snapshot.db
PASS: Database inputs checked.


### Cell 30 — Define transactional snapshot loading

**Purpose and method:** Create typed tables, primary/foreign keys, indexes and reporting/quarantine views. Replace this dedicated database snapshot inside one transaction; rollback protects against incomplete loads.

**Expected result / interpretation:** A loader that closes connections and checks counts before committing. This is not historical batch storage.


In [30]:
def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'


def sqlite_type(series):
    if pd.api.types.is_bool_dtype(series.dtype):
        return "INTEGER"
    if pd.api.types.is_integer_dtype(series.dtype):
        return "INTEGER"
    if pd.api.types.is_float_dtype(series.dtype):
        return "REAL"
    return "TEXT"


def insert_dataframe(connection, table_name, frame):
    columns_sql = ", ".join(
        quote_identifier(column) for column in frame.columns
    )
    placeholders = ", ".join(["?"] * len(frame.columns))

    statement = (
        f"INSERT INTO {quote_identifier(table_name)} "
        f"({columns_sql}) VALUES ({placeholders})"
    )

    # Convert pandas nullable values into SQL NULL.
    sql_ready = frame.astype(object).where(frame.notna(), None)

    connection.executemany(
        statement,
        sql_ready.itertuples(index=False, name=None)
    )


def load_snapshot(db_path, records, rules, exceptions, metadata):
    connection = sqlite3.connect(db_path)
    connection.execute("PRAGMA foreign_keys = ON")

    try:
        connection.execute("BEGIN")

        connection.execute("""
            CREATE TABLE IF NOT EXISTS source_metadata (
                sha256 TEXT PRIMARY KEY,
                filename TEXT NOT NULL,
                file_size_bytes INTEGER NOT NULL,
                row_count INTEGER NOT NULL,
                column_count INTEGER NOT NULL
            )
        """)

        connection.execute("""
            CREATE TABLE IF NOT EXISTS validation_rules (
                rule_id TEXT PRIMARY KEY,
                severity TEXT NOT NULL
                    CHECK (severity IN ('ERROR', 'WARNING')),
                field TEXT NOT NULL,
                description TEXT NOT NULL
            )
        """)

        column_definitions = []

        for column in records.columns:
            definition = (
                f"{quote_identifier(column)} "
                f"{sqlite_type(records[column])}"
            )

            if column in {
                "source_file_sha256",
                "source_record_number",
                "validation_status",
            }:
                definition += " NOT NULL"

            column_definitions.append(definition)

        column_definitions.extend([
            """
            PRIMARY KEY (
                source_file_sha256,
                source_record_number
            )
            """,
            """
            FOREIGN KEY (source_file_sha256)
                REFERENCES source_metadata(sha256)
            """,
            """
            CHECK (
                validation_status IN ('PASS', 'REVIEW', 'ERROR')
            )
            """,
        ])

        connection.execute(
            "CREATE TABLE IF NOT EXISTS mortgage_records ("
            + ", ".join(column_definitions)
            + ")"
        )

        connection.execute("""
            CREATE TABLE IF NOT EXISTS validation_exceptions (
                source_file_sha256 TEXT NOT NULL,
                source_record_number INTEGER NOT NULL,
                rule_id TEXT NOT NULL,
                severity TEXT NOT NULL
                    CHECK (severity IN ('ERROR', 'WARNING')),
                field TEXT NOT NULL,
                raw_value TEXT,
                description TEXT NOT NULL,

                PRIMARY KEY (
                    source_file_sha256,
                    source_record_number,
                    rule_id
                ),

                FOREIGN KEY (
                    source_file_sha256,
                    source_record_number
                ) REFERENCES mortgage_records (
                    source_file_sha256,
                    source_record_number
                ),

                FOREIGN KEY (rule_id)
                    REFERENCES validation_rules(rule_id)
            )
        """)

        # This database stores ONE snapshot.
        # Remove child rows before their referenced parent rows.
        for table in [
            "validation_exceptions",
            "mortgage_records",
            "validation_rules",
            "source_metadata",
        ]:
            connection.execute(
                f"DELETE FROM {quote_identifier(table)}"
            )

        insert_dataframe(
            connection,
            "source_metadata",
            pd.DataFrame([metadata])
        )
        insert_dataframe(connection, "validation_rules", rules)
        insert_dataframe(connection, "mortgage_records", records)
        insert_dataframe(
            connection, "validation_exceptions", exceptions
        )

        connection.execute("""
            CREATE INDEX IF NOT EXISTS idx_records_status
            ON mortgage_records(validation_status)
        """)

        connection.execute("""
            CREATE INDEX IF NOT EXISTS idx_records_lender
            ON mortgage_records(lei)
        """)

        connection.execute("""
            CREATE VIEW IF NOT EXISTS reporting_records AS
            SELECT *
            FROM mortgage_records
            WHERE validation_status IN ('PASS', 'REVIEW')
        """)

        connection.execute("""
            CREATE VIEW IF NOT EXISTS quarantined_records AS
            SELECT *
            FROM mortgage_records
            WHERE validation_status = 'ERROR'
        """)

        counts = {}

        expected_counts = {
            "source_metadata": 1,
            "mortgage_records": len(records),
            "validation_rules": len(rules),
            "validation_exceptions": len(exceptions),
        }

        for table, expected in expected_counts.items():
            actual = connection.execute(
                f"SELECT COUNT(*) FROM {quote_identifier(table)}"
            ).fetchone()[0]

            if actual != expected:
                raise ValueError(
                    f"{table}: expected {expected}, found {actual}"
                )

            counts[table] = actual

        if connection.execute("PRAGMA foreign_key_check").fetchall():
            raise ValueError("Foreign-key validation failed.")

        connection.commit()
        return counts

    except Exception:
        connection.rollback()
        raise

    finally:
        connection.close()


print("Transactional snapshot loader ready.")

Transactional snapshot loader ready.


### Cell 31 — Verify repeatable loading

**Purpose and method:** Load the same snapshot twice and compare counts. This deliberately exercises rerun behavior.

**Expected result / interpretation:** 1 metadata row, 82,691 records, 16 rules and 1,646 exceptions after either load.


In [31]:
first_load_counts = load_snapshot(
    DB_PATH,
    prepared_df,
    rule_register,
    exceptions_df,
    source_metadata
)

second_load_counts = load_snapshot(
    DB_PATH,
    prepared_df,
    rule_register,
    exceptions_df,
    source_metadata
)

assert first_load_counts == second_load_counts

display(
    pd.DataFrame(
        second_load_counts.items(),
        columns=["table", "records"]
    )
)

print("PASS: Both loads produced the same record counts.")
print("PASS: Rerunning did not accumulate records.")

,table,records
0,source_metadata,1
1,mortgage_records,82691
2,validation_rules,16
3,validation_exceptions,1646


PASS: Both loads produced the same record counts.
PASS: Rerunning did not accumulate records.


### Cell 32 — Query stored validation results

**Purpose and method:** Read status and warning summaries using SQL. GROUP BY returns only categories present, so an absent ERROR row means zero records in that category.

**Expected result / interpretation:** Database summaries matching the prepared data.


In [32]:
def query_database(sql, params=None):
    with sqlite3.connect(DB_PATH) as connection:
        return pd.read_sql_query(
            sql,
            connection,
            params=params
        )


sql_status_summary = query_database("""
    SELECT
        validation_status,
        COUNT(*) AS records
    FROM mortgage_records
    GROUP BY validation_status
    ORDER BY validation_status
""")

display(sql_status_summary)


sql_exception_summary = query_database("""
    SELECT
        severity,
        COUNT(*) AS exception_events
    FROM validation_exceptions
    GROUP BY severity
""")

display(sql_exception_summary)


sql_review_count = query_database("""
    SELECT COUNT(*) AS distinct_records_with_warnings
    FROM (
        SELECT DISTINCT
            source_file_sha256,
            source_record_number
        FROM validation_exceptions
        WHERE severity = 'WARNING'
    )
""")

display(sql_review_count)

,validation_status,records
0,PASS,81147
1,REVIEW,1544


,severity,exception_events
0,WARNING,1646


,distinct_records_with_warnings
0,1544


### Cell 33 — Verify database values and integrity

**Purpose and method:** Compare selected source/derived values after SQL storage and run integrity and foreign-key checks. Values for every column are not individually compared here.

**Expected result / interpretation:** All selected values match; database integrity and references pass.


In [33]:
verification_columns = [
    "source_record_number",
    "loan_amount",
    "loan_amount_numeric",
    "income",
    "income_numeric",
    "income_parse_status",
    "interest_rate",
    "interest_rate_numeric",
    "interest_rate_parse_status",
    "debt_to_income_ratio",
    "dti_representation",
    "validation_status",
]

selected_columns = ", ".join(
    quote_identifier(column)
    for column in verification_columns
)

stored_values = query_database(
    f"""
    SELECT {selected_columns}
    FROM mortgage_records
    ORDER BY source_record_number
    """
)

expected_values = (
    prepared_df[verification_columns]
    .sort_values("source_record_number")
    .reset_index(drop=True)
)

# Normalize missing representations for comparison.
for column in verification_columns:
    expected = expected_values[column].astype(object)
    actual = stored_values[column].astype(object)

    expected = expected.where(expected.notna(), None)
    actual = actual.where(actual.notna(), None)

    pd.testing.assert_series_equal(
        actual,
        expected,
        check_dtype=False,
        check_names=False
    )

with sqlite3.connect(DB_PATH) as connection:
    integrity_result = connection.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]

    foreign_key_issues = connection.execute(
        "PRAGMA foreign_key_check"
    ).fetchall()

assert integrity_result == "ok"
assert not foreign_key_issues

print("PASS: Selected stored values match the prepared data.")
print("PASS: SQLite integrity check passed.")
print("PASS: No foreign-key violations.")
print(f"Database size: {DB_PATH.stat().st_size / 1024**2:.2f} MB")

PASS: Selected stored values match the prepared data.
PASS: SQLite integrity check passed.
PASS: No foreign-key violations.
Database size: 57.73 MB


## 6. SQL business reporting

### Cell 34 — Report all outcomes

**Purpose and method:** Count each outcome from the reporting view and reconcile the total. The denominator includes all reporting-eligible outcomes.

**Expected result / interpretation:** A report suitable for describing dataset composition, not default or approval probability.


In [34]:
outcome_report = query_database("""
    SELECT
        action_taken AS action_code,
        action_label AS outcome,
        COUNT(*) AS records,
        ROUND(
            100.0 * COUNT(*)
            / (SELECT COUNT(*) FROM reporting_records),
            2
        ) AS share_of_records_pct
    FROM reporting_records
    GROUP BY action_taken, action_label
    ORDER BY records DESC
""")

display(outcome_report)

assert outcome_report["records"].sum() == len(reporting_df)

print("PASS: Outcome report reconciles to reporting-eligible records.")

,action_code,outcome,records,share_of_records_pct
0,1,Loan originated,33869,40.96
1,3,Application denied,20868,25.24
2,4,Application withdrawn,11709,14.16
3,6,Purchased loan,7593,9.18
4,5,File closed for incompleteness,5663,6.85
5,2,Application approved but not accepted,2908,3.52
6,8,Preapproval request approved but not accepted,63,0.08
7,7,Preapproval request denied,18,0.02


PASS: Outcome report reconciles to reporting-eligible records.


### Cell 35 — Rank lenders across all originations

**Purpose and method:** Filter action code 1, then aggregate counts, reported amount and review flags by LEI. This includes multiple property categories.

**Expected result / interpretation:** 610 lenders and 33,869 originations. Public amounts are privacy-modified.


In [35]:
lender_report = query_database("""
    SELECT
        lei AS lender_identifier,
        COUNT(*) AS originated_loans,
        SUM(loan_amount_numeric) AS reported_loan_amount_sum,
        AVG(loan_amount_numeric) AS mean_reported_loan_amount,
        SUM(
            CASE WHEN validation_status = 'REVIEW'
                 THEN 1 ELSE 0 END
        ) AS loans_with_review_flags
    FROM reporting_records
    WHERE action_taken = '1'
    GROUP BY lei
    ORDER BY reported_loan_amount_sum DESC, lei
""")

display(lender_report.head(15))

print(f"Lenders with originated loans: {len(lender_report):,}")
print(f"Originated loans: {lender_report['originated_loans'].sum():,}")

,lender_identifier,originated_loans,reported_loan_amount_sum,mean_reported_loan_amount,loans_with_review_flags
0,549300HW662MN1WU8550,5044,2.290890e+09,4.541812e+05,19
1,7H6GLXDRUGQFU57RNE97,911,1.268475e+09,1.392398e+06,0
2,549300FGXN1K3HLB1R50,1986,7.075900e+08,3.562890e+05,0
3,2549005SIVTHG14U2905,1420,5.560700e+08,3.915986e+05,8
4,B4TYDEB6GKMZO031MB27,722,3.951700e+08,5.473269e+05,1
5,549300H3IZO24NSOO931,777,3.864250e+08,4.973295e+05,0
6,254900ZFWS2106HWPH46,830,3.841300e+08,4.628072e+05,32
7,5493007Z7FTUKQGFMZ98,3,3.635250e+08,1.211750e+08,0
8,GDN5JXQ4P5NZE28ZRC90,176,3.102900e+08,1.763011e+06,0
9,KB1H1DSPRFMYMCUFXT09,201,2.876050e+08,1.430871e+06,0


Lenders with originated loans: 610
Originated loans: 33,869


### Cell 36 — Compare loan purposes

**Purpose and method:** Group all records by purpose while separately counting originations/denials and summing origination amounts. Other outcomes explain why the two counts do not sum to total records.

**Expected result / interpretation:** Six purpose groups that reconcile to the reporting population.


In [36]:
purpose_report = query_database("""
    SELECT
        loan_purpose AS purpose_code,

        CASE loan_purpose
            WHEN '1'  THEN 'Home purchase'
            WHEN '2'  THEN 'Home improvement'
            WHEN '31' THEN 'Refinancing'
            WHEN '32' THEN 'Cash-out refinancing'
            WHEN '4'  THEN 'Other purpose'
            WHEN '5'  THEN 'Not applicable'
            ELSE 'Unrecognized code - review'
        END AS loan_purpose_label,

        COUNT(*) AS total_records,

        SUM(
            CASE WHEN action_taken = '1'
                 THEN 1 ELSE 0 END
        ) AS originated_loans,

        SUM(
            CASE WHEN action_taken = '3'
                 THEN 1 ELSE 0 END
        ) AS denied_applications,

        SUM(
            CASE WHEN action_taken = '1'
                 THEN loan_amount_numeric ELSE 0 END
        ) AS originated_reported_amount_sum

    FROM reporting_records
    GROUP BY loan_purpose
    ORDER BY total_records DESC
""")

display(purpose_report)

assert purpose_report["total_records"].sum() == len(reporting_df)

print("PASS: Loan-purpose counts reconcile.")

,purpose_code,loan_purpose_label,total_records,originated_loans,denied_applications,originated_reported_amount_sum
0,1,Home purchase,39089,19558,5838,1.176833e+10
1,32,Cash-out refinancing,15557,5894,4119,2.249000e+09
2,4,Other purpose,11057,2916,4762,8.590300e+08
3,2,Home improvement,9686,2871,4121,5.336950e+08
4,31,Refinancing,7167,2603,2021,2.548195e+09
5,5,Not applicable,135,27,7,1.576500e+07


PASS: Loan-purpose counts reconcile.


### Cell 37 — Report rule-level review work

**Purpose and method:** Use a SQL LEFT JOIN to retain zero-finding rules alongside triggered rules. These counts measure events, not confirmed data defects.

**Expected result / interpretation:** A 16-rule report totalling 1,646 events.


In [37]:
quality_report = query_database("""
    SELECT
        r.rule_id,
        r.severity,
        r.field,
        r.description,
        COUNT(e.rule_id) AS exception_events
    FROM validation_rules AS r
    LEFT JOIN validation_exceptions AS e
        ON r.rule_id = e.rule_id
    GROUP BY
        r.rule_id,
        r.severity,
        r.field,
        r.description
    ORDER BY
        exception_events DESC,
        r.rule_id
""")

display(quality_report)

assert quality_report["exception_events"].sum() == len(exceptions_df)

print("PASS: Quality report reconciles to the exception table.")

,rule_id,severity,field,description,exception_events
0,WARN_ZERO_INTEREST,WARNING,interest_rate,Reported interest rate is zero; investigate without chan...,1124
1,WARN_EXACT_ROW_MATCH,WARNING,__entire_source_row__,Entire source row matches another row; retain pending re...,415
2,WARN_NEGATIVE_INCOME,WARNING,income,Reported income is negative; investigate without changin...,101
3,WARN_STATE_UNAVAILABLE,WARNING,state_code,State is blank or NA; review against the county.,6
4,ERR_ACTION_CODE,ERROR,action_taken,Action code must be one of the documented codes 1 throug...,0
5,ERR_COUNTY_SCOPE,ERROR,county_code,County must match this project's 12086 scope.,0
6,ERR_DTI_REPRESENTATION,ERROR,debt_to_income_ratio,"DTI must use a recognized numeric, range, or special rep...",0
7,ERR_LEI_FORMAT,ERROR,lei,Lender identifier must contain 20 uppercase letters or d...,0
8,ERR_LOAN_AMOUNT,ERROR,loan_amount,Loan amount must parse as a finite number greater than z...,0
9,ERR_PARSE_INCOME,ERROR,income,"Unexpected numeric text or a non-finite value. Blank, NA...",0


PASS: Quality report reconciles to the exception table.


### Cell 38 — Reconcile the management overview

**Purpose and method:** Summarize record status and independently compare SQL originations and amount totals against pandas.

**Expected result / interpretation:** 82,691 records, 737 lenders and 1.87% REVIEW, with consistent aggregate totals.


In [38]:
overview_report = query_database("""
    SELECT
        COUNT(*) AS total_stored_records,

        COUNT(DISTINCT lei) AS distinct_lender_identifiers,

        SUM(
            CASE WHEN validation_status = 'PASS'
                 THEN 1 ELSE 0 END
        ) AS pass_records,

        SUM(
            CASE WHEN validation_status = 'REVIEW'
                 THEN 1 ELSE 0 END
        ) AS review_records,

        SUM(
            CASE WHEN validation_status = 'ERROR'
                 THEN 1 ELSE 0 END
        ) AS quarantined_records,

        ROUND(
            100.0 * SUM(
                CASE WHEN validation_status = 'REVIEW'
                     THEN 1 ELSE 0 END
            ) / NULLIF(COUNT(*), 0),
            2
        ) AS review_records_pct

    FROM mortgage_records
""")

display(overview_report)

# Independent comparison with the prepared DataFrame.
originated_mask = reporting_df["action_taken"].eq("1")

expected_originations = int(originated_mask.sum())

expected_originated_amount = float(
    reporting_df.loc[
        originated_mask,
        "loan_amount_numeric"
    ].sum()
)

assert lender_report["originated_loans"].sum() == expected_originations
assert purpose_report["originated_loans"].sum() == expected_originations

np.testing.assert_allclose(
    lender_report["reported_loan_amount_sum"].sum(),
    expected_originated_amount,
    rtol=1e-12,
    atol=0.01
)

np.testing.assert_allclose(
    purpose_report["originated_reported_amount_sum"].sum(),
    expected_originated_amount,
    rtol=1e-12,
    atol=0.01
)

assert overview_report.loc[0, "total_stored_records"] == len(raw_df)

print("PASS: SQL originations reconcile to the prepared data.")
print("PASS: Lender and purpose amount totals reconcile.")
print("PASS: Overview record count matches the source.")

,total_stored_records,distinct_lender_identifiers,pass_records,review_records,quarantined_records,review_records_pct
0,82691,737,81147,1544,0,1.87


PASS: SQL originations reconcile to the prepared data.
PASS: Lender and purpose amount totals reconcile.
PASS: Overview record count matches the source.


## 7. Investigation and property segmentation

### Cell 39 — Inspect unusually large originations

**Purpose and method:** Display large transactions with property categories and unit counts. Investigate context before classifying large amounts as errors.

**Expected result / interpretation:** Most largest transactions are multifamily; record 1450 has a separate ratio review candidate.


In [39]:
largest_originations = query_database("""
    SELECT
        source_record_number,
        lei AS lender_identifier,
        loan_amount_numeric AS reported_loan_amount,
        property_value_numeric AS reported_property_value,
        derived_dwelling_category,
        total_units,
        occupancy_type,
        business_or_commercial_purpose,
        loan_purpose,
        loan_to_value_ratio,
        validation_status
    FROM reporting_records
    WHERE action_taken = '1'
    ORDER BY loan_amount_numeric DESC
    LIMIT 15
""")

display(
    largest_originations.style.format({
        "reported_loan_amount": "{:,.0f}",
        "reported_property_value": "{:,.0f}",
    }, na_rep="Not available")
)

,source_record_number,lender_identifier,reported_loan_amount,reported_property_value,derived_dwelling_category,total_units,occupancy_type,business_or_commercial_purpose,loan_purpose,loan_to_value_ratio,validation_status
0,81912,5493007Z7FTUKQGFMZ98,"334,595,000","839,005,000",Multifamily:Site-Built,>149,3,1,31,39.88,PASS
1,1450,7H6GLXDRUGQFU57RNE97,"333,965,000","405,405,000",Multifamily:Site-Built,>149,3,1,1,727.68,PASS
2,1449,7H6GLXDRUGQFU57RNE97,"195,005,000","339,005,000",Multifamily:Site-Built,>149,3,1,31,57.52,PASS
3,3232,B2S31CFVSWTN3FR00Q90,"147,705,000","217,295,000",Multifamily:Site-Built,>149,3,1,31,67.97,PASS
4,45856,AD6GFRVSDT01YPT1CS68,"71,105,000","125,805,000",Multifamily:Site-Built,>149,3,1,1,56.52,PASS
5,13715,549300JGMQJ4R419LR70,"71,095,000",Not available,Multifamily:Site-Built,>149,3,1,31,NA,PASS
6,10450,KB1H1DSPRFMYMCUFXT09,"67,445,000","115,805,000",Multifamily:Site-Built,>149,3,1,31,58.24,PASS
7,56887,254900YA1AQXNM8QVZ06,"62,755,000","93,705,000",Multifamily:Site-Built,>149,3,1,31,67.000000,PASS
8,10454,KB1H1DSPRFMYMCUFXT09,"52,465,000","203,235,000",Multifamily:Site-Built,>149,3,1,31,53,PASS
9,50569,549300DH8EI64ITBY388,"49,635,000","78,605,000",Multifamily:Site-Built,>149,3,1,1,63.15,PASS


### Cell 40 — Explain property-segment differences

**Purpose and method:** Group originations by dwelling category and reconcile counts and amounts to the full origination population.

**Expected result / interpretation:** 33,689 site-built 1-4-unit, 128 multifamily site-built and 52 manufactured 1-4-unit originations.


In [40]:
dwelling_report = query_database("""
    SELECT
        derived_dwelling_category AS dwelling_category,
        COUNT(*) AS originated_loans,
        SUM(loan_amount_numeric) AS reported_amount_sum,
        AVG(loan_amount_numeric) AS mean_reported_amount,
        MAX(loan_amount_numeric) AS largest_reported_amount
    FROM reporting_records
    WHERE action_taken = '1'
    GROUP BY derived_dwelling_category
    ORDER BY reported_amount_sum DESC
""")

display(
    dwelling_report.style.format({
        "originated_loans": "{:,}",
        "reported_amount_sum": "{:,.0f}",
        "mean_reported_amount": "{:,.0f}",
        "largest_reported_amount": "{:,.0f}",
    })
)

assert dwelling_report["originated_loans"].sum() == expected_originations

np.testing.assert_allclose(
    dwelling_report["reported_amount_sum"].sum(),
    expected_originated_amount,
    rtol=1e-12,
    atol=0.01
)

print("PASS: Dwelling-category totals reconcile to all originations.")

,dwelling_category,originated_loans,reported_amount_sum,mean_reported_amount,largest_reported_amount
0,Single Family (1-4 Units):Site-Built,"33,689","15,904,045,000","472,084","35,005,000"
1,Multifamily:Site-Built,128,"2,059,750,000","16,091,797","334,595,000"
2,Single Family (1-4 Units):Manufactured,52,"10,220,000","196,538","515,000"


PASS: Dwelling-category totals reconcile to all originations.


### Cell 41 — Investigate a specific lender

**Purpose and method:** Use a parameterized SQL query to inspect the three large originations of the selected LEI. This investigation is specific to the current snapshot.

**Expected result / interpretation:** Three multifamily transactions totalling USD 363,525,000; size alone does not justify removal.


In [41]:
lender_to_review = "5493007Z7FTUKQGFMZ98"

lender_detail = query_database("""
    SELECT
        source_record_number,
        loan_amount_numeric AS reported_loan_amount,
        property_value_numeric AS reported_property_value,
        derived_dwelling_category,
        total_units,
        loan_purpose,
        business_or_commercial_purpose,
        validation_status
    FROM reporting_records
    WHERE action_taken = '1'
      AND lei = ?
    ORDER BY loan_amount_numeric DESC
""", params=(lender_to_review,))

display(
    lender_detail.style.format({
        "reported_loan_amount": "{:,.0f}",
        "reported_property_value": "{:,.0f}",
    }, na_rep="Not available")
)

print(f"Originations for this lender: {len(lender_detail):,}")
print(
    "Reported amount total:",
    f"{lender_detail['reported_loan_amount'].sum():,.0f}"
)

,source_record_number,reported_loan_amount,reported_property_value,derived_dwelling_category,total_units,loan_purpose,business_or_commercial_purpose,validation_status
0,81912,"334,595,000","839,005,000",Multifamily:Site-Built,>149,31,1,PASS
1,81911,"24,025,000","41,905,000",Multifamily:Site-Built,>149,31,1,PASS
2,81913,"4,905,000","6,655,000",Multifamily:Site-Built,25-49,31,1,PASS


Originations for this lender: 3
Reported amount total: 363,525,000


### Cell 42 — Build the full property-segment report

**Purpose and method:** Preserve all categories and compare record counts, originations, reported amounts and warning counts.

**Expected result / interpretation:** Three segments that reconcile to the full reporting dataset.


In [42]:
segment_report = query_database("""
    SELECT
        derived_dwelling_category AS dwelling_category,
        COUNT(*) AS total_records,

        SUM(
            CASE WHEN action_taken = '1'
                 THEN 1 ELSE 0 END
        ) AS originated_loans,

        SUM(
            CASE WHEN action_taken = '1'
                 THEN loan_amount_numeric ELSE 0 END
        ) AS originated_reported_amount_sum,

        SUM(
            CASE WHEN validation_status = 'REVIEW'
                 THEN 1 ELSE 0 END
        ) AS records_with_review_flags

    FROM reporting_records
    GROUP BY derived_dwelling_category
    ORDER BY total_records DESC
""")

display(
    segment_report.style.format({
        "total_records": "{:,}",
        "originated_loans": "{:,}",
        "originated_reported_amount_sum": "{:,.0f}",
        "records_with_review_flags": "{:,}",
    })
)

assert segment_report["total_records"].sum() == len(reporting_df)

assert (
    segment_report["originated_loans"].sum()
    == expected_originations
)

np.testing.assert_allclose(
    segment_report["originated_reported_amount_sum"].sum(),
    expected_originated_amount,
    rtol=1e-12,
    atol=0.01
)

print("PASS: Property segments reconcile to the full reporting dataset.")

,dwelling_category,total_records,originated_loans,originated_reported_amount_sum,records_with_review_flags
0,Single Family (1-4 Units):Site-Built,"82,176","33,689","15,904,045,000","1,543"
1,Single Family (1-4 Units):Manufactured,309,52,"10,220,000",1
2,Multifamily:Site-Built,206,128,"2,059,750,000",0


PASS: Property segments reconcile to the full reporting dataset.


### Cell 43 — Create the focused lender comparison

**Purpose and method:** Filter to site-built properties with 1-4 units and originations only, then reconcile with the same pandas subset. This scope can include investment properties.

**Expected result / interpretation:** 599 lenders and 33,689 originations; a separate, explicitly scoped lender report.


In [43]:
FOCUS_CATEGORY = "Single Family (1-4 Units):Site-Built"

focused_lender_report = query_database("""
    SELECT
        lei AS lender_identifier,
        COUNT(*) AS originated_loans,
        SUM(loan_amount_numeric) AS reported_loan_amount_sum,
        AVG(loan_amount_numeric) AS mean_reported_loan_amount,

        SUM(
            CASE WHEN validation_status = 'REVIEW'
                 THEN 1 ELSE 0 END
        ) AS loans_with_review_flags

    FROM reporting_records
    WHERE action_taken = '1'
      AND derived_dwelling_category = ?

    GROUP BY lei
    ORDER BY reported_loan_amount_sum DESC, lei
""", params=(FOCUS_CATEGORY,))

display(
    focused_lender_report.head(15).style.format({
        "originated_loans": "{:,}",
        "reported_loan_amount_sum": "{:,.0f}",
        "mean_reported_loan_amount": "{:,.0f}",
        "loans_with_review_flags": "{:,}",
    })
)

focus_mask = (
    reporting_df["action_taken"].eq("1")
    & reporting_df["derived_dwelling_category"].eq(FOCUS_CATEGORY)
)

assert (
    focused_lender_report["originated_loans"].sum()
    == int(focus_mask.sum())
)

np.testing.assert_allclose(
    focused_lender_report["reported_loan_amount_sum"].sum(),
    float(
        reporting_df.loc[focus_mask, "loan_amount_numeric"].sum()
    ),
    rtol=1e-12,
    atol=0.01
)

print("PASS: Focused lender report reconciles to its defined subset.")

,lender_identifier,originated_loans,reported_loan_amount_sum,mean_reported_loan_amount,loans_with_review_flags
0,549300HW662MN1WU8550,"5,034","2,288,690,000","454,646",19
1,7H6GLXDRUGQFU57RNE97,909,"739,505,000","813,537",0
2,549300FGXN1K3HLB1R50,"1,984","707,200,000","356,452",0
3,2549005SIVTHG14U2905,"1,412","551,790,000","390,786",8
4,B4TYDEB6GKMZO031MB27,722,"395,170,000","547,327",1
5,549300H3IZO24NSOO931,777,"386,425,000","497,329",0
6,254900ZFWS2106HWPH46,829,"383,865,000","463,046",32
7,549300GS0W0TEUQS8571,137,"257,595,000","1,880,255",0
8,E57ODZWZ7FF32TWEFA76,177,"231,975,000","1,310,593",0
9,5493006Q0EMYD3OJ8N13,361,"228,885,000","634,030",8


PASS: Focused lender report reconciles to its defined subset.


## 8. Report delivery and backup

### Cell 44 — Export CSV report checkpoints

**Purpose and method:** Write eight report tables to the reports directory. This exports summaries and exception details, not a replacement for the original input.

**Expected result / interpretation:** Eight CSV files and an export manifest. Files under /content are temporary.


In [44]:
REPORT_DIRECTORY = Path("/content/mortgage_project/reports")
REPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)

report_tables = {
    "overview": overview_report,
    "outcomes": outcome_report,
    "loan_purposes": purpose_report,
    "property_segments": segment_report,
    "lenders_all_originations": lender_report,
    "lenders_1_to_4_unit_site_built": focused_lender_report,
    "validation_rule_results": quality_report,
    "validation_exception_details": exceptions_df,
}

export_manifest = []

for report_name, report_df in report_tables.items():
    output_path = REPORT_DIRECTORY / f"{report_name}.csv"

    report_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8"
    )

    export_manifest.append({
        "report": report_name,
        "rows": len(report_df),
        "columns": len(report_df.columns),
        "filename": output_path.name,
    })

display(pd.DataFrame(export_manifest))

print("Report tables saved to:", REPORT_DIRECTORY)
print("These Colab files remain temporary until downloaded or copied to Drive.")

,report,rows,columns,filename
0,overview,1,6,overview.csv
1,outcomes,8,4,outcomes.csv
2,loan_purposes,6,6,loan_purposes.csv
3,property_segments,3,5,property_segments.csv
4,lenders_all_originations,610,5,lenders_all_originations.csv
5,lenders_1_to_4_unit_site_built,599,5,lenders_1_to_4_unit_site_built.csv
6,validation_rule_results,16,5,validation_rule_results.csv
7,validation_exception_details,1646,7,validation_exception_details.csv


Report tables saved to: /content/mortgage_project/reports
These Colab files remain temporary until downloaded or copied to Drive.


### Cell 45 — Import Excel reporting tools

**Purpose and method:** Load OpenPyXL objects for workbook formatting, tables and charts, and set the output filename.

**Expected result / interpretation:** The Excel output destination is ready.


In [45]:
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.chart import BarChart, Reference
from openpyxl.utils import get_column_letter

EXCEL_PATH = REPORT_DIRECTORY / "Mortgage_Data_Quality_Report_2024.xlsx"

print("Excel tools ready.")

Excel tools ready.


### Cell 46 — Construct the Excel report

**Purpose and method:** Populate eight sheets with scope labels, typed values, tables, formats and frozen headers. Preserve formula-like source strings as text and add an outcome chart.

**Expected result / interpretation:** Formatted report sheets with a horizontal bar chart showing outcomes on the vertical axis and record counts on the horizontal axis.

In [46]:
# Cell 46 — Construct the Excel report

workbook = Workbook()
workbook.remove(workbook.active)

NAVY = "17365D"
WHITE = "FFFFFF"

sheet_definitions = [
    (
        "Overview",
        overview_report,
        "All stored records: 2024, Miami-Dade County (12086)."
    ),
    (
        "Outcomes",
        outcome_report,
        "All reporting-eligible records; percentages use all such records."
    ),
    (
        "Loan Purposes",
        purpose_report,
        "All reporting-eligible records; amount totals include originations only."
    ),
    (
        "Property Segments",
        segment_report,
        "All property categories retained; amount totals include originations only."
    ),
    (
        "Lenders All",
        lender_report,
        "Originations across all property categories; lenders identified by LEI."
    ),
    (
        "Lenders 1-4 Site Built",
        focused_lender_report,
        "Originations for 1-4-unit site-built properties, including investment uses."
    ),
    (
        "Quality Rules",
        quality_report,
        "Implemented rule results; warning events may overlap on the same record."
    ),
    (
        "Exceptions",
        exceptions_df,
        "One row per triggered rule and source record; warnings are review items."
    ),
]

money_columns = {
    "reported_loan_amount_sum",
    "mean_reported_loan_amount",
    "originated_reported_amount_sum",
}

text_columns = {
    "source_file_sha256",
    "rule_id",
    "severity",
    "field",
    "raw_value",
    "description",
    "lender_identifier",
    "action_code",
    "purpose_code",
    "outcome",
    "loan_purpose_label",
    "dwelling_category",
}

for table_number, (sheet_name, frame, scope) in enumerate(
    sheet_definitions,
    start=1
):
    sheet = workbook.create_sheet(sheet_name)
    number_of_columns = len(frame.columns)

    # Report title.
    sheet.merge_cells(
        start_row=1,
        start_column=1,
        end_row=1,
        end_column=number_of_columns
    )

    sheet.cell(
        1,
        1,
        f"Mortgage Data Quality Report — {sheet_name}"
    )

    sheet.cell(1, 1).font = Font(
        size=16,
        bold=True,
        color=NAVY
    )

    sheet.row_dimensions[1].height = 28

    # Scope description.
    sheet.merge_cells(
        start_row=2,
        start_column=1,
        end_row=2,
        end_column=number_of_columns
    )

    sheet.cell(2, 1, scope)

    sheet.cell(2, 1).alignment = Alignment(
        wrap_text=True,
        vertical="top"
    )

    sheet.row_dimensions[2].height = 32

    # Table headers.
    for column_number, column_name in enumerate(
        frame.columns,
        start=1
    ):
        cell = sheet.cell(4, column_number, column_name)

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor=NAVY
        )

        cell.font = Font(
            bold=True,
            color=WHITE
        )

        cell.alignment = Alignment(
            wrap_text=True,
            vertical="center"
        )

    sheet.row_dimensions[4].height = 42

    # Table data.
    for row_number, values in enumerate(
        frame.itertuples(index=False, name=None),
        start=5
    ):
        for column_number, value in enumerate(values, start=1):
            column_name = frame.columns[column_number - 1]

            if pd.isna(value):
                value = None
            elif isinstance(value, np.generic):
                value = value.item()

            cell = sheet.cell(
                row_number,
                column_number,
                value
            )

            # Preserve source strings as text, including formula-like values.
            if isinstance(value, str):
                cell.data_type = "s"

            if column_name in money_columns:
                cell.number_format = '"$"#,##0.00'

            elif column_name.endswith("_pct"):
                # Reports store percentages as 1.87 rather than 0.0187.
                cell.number_format = '0.00"%"'

            elif isinstance(value, (int, float)):
                cell.number_format = "#,##0"

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=column_name in {
                    "description",
                    "outcome",
                    "loan_purpose_label",
                    "dwelling_category",
                }
            )

    # Add an Excel table with filter controls.
    last_row = 4 + len(frame)
    last_column_letter = get_column_letter(number_of_columns)

    if len(frame):
        excel_table = Table(
            displayName=f"ReportTable{table_number}",
            ref=f"A4:{last_column_letter}{last_row}"
        )

        excel_table.tableStyleInfo = TableStyleInfo(
            name="TableStyleMedium2",
            showRowStripes=True
        )

        sheet.add_table(excel_table)

    # Column widths.
    for column_number, column_name in enumerate(
        frame.columns,
        start=1
    ):
        width = 23

        if column_name == "source_file_sha256":
            width = 68

        elif column_name == "description":
            width = 65

        elif column_name in {
            "outcome",
            "dwelling_category",
            "rule_id",
            "raw_value",
        }:
            width = 43

        elif column_name == "lender_identifier":
            width = 27

        elif column_name in text_columns:
            width = 30

        sheet.column_dimensions[
            get_column_letter(column_number)
        ].width = width

    # Navigation and printing settings.
    sheet.freeze_panes = "A5"
    sheet.sheet_view.showGridLines = False
    sheet.sheet_properties.pageSetUpPr.fitToPage = True

    sheet.page_setup.orientation = "landscape"
    sheet.page_setup.paperSize = sheet.PAPERSIZE_A4
    sheet.page_setup.fitToWidth = 1
    sheet.page_setup.fitToHeight = 0

    sheet.print_title_rows = "1:4"

# Add the outcome chart.
outcomes_sheet = workbook["Outcomes"]

chart = BarChart()
chart.type = "bar"
chart.title = "Reported outcomes"

# Correct axis titles for an OpenPyXL horizontal bar chart:
# Category axis is displayed vertically.
# Numeric axis is displayed horizontally.
chart.x_axis.title = "Outcome"
chart.y_axis.title = "Records"

chart.legend = None
chart.height = 12
chart.width = 23

chart.add_data(
    Reference(
        outcomes_sheet,
        min_col=3,
        min_row=4,
        max_row=4 + len(outcome_report)
    ),
    titles_from_data=True
)

chart.set_categories(
    Reference(
        outcomes_sheet,
        min_col=2,
        min_row=5,
        max_row=4 + len(outcome_report)
    )
)

outcomes_sheet.add_chart(chart, "F4")

print("Report sheets and outcome chart prepared.")

Report sheets and outcome chart prepared.


### Cell 47 — Document, save and verify the workbook

**Purpose and method:** Calculate warning-event and distinct-record counts from the exception table. Add source details, reporting definitions and limitations. Include the record-1450 investigation only when the source fingerprint matches the investigated file. Save the workbook, reopen it and compare the exported tables with their source DataFrames.

**Expected result / interpretation:** A workbook with nine sheets, including one “Source and Notes” sheet. Saved headers, row counts, table values and documentation pass verification. Rerunning this cell replaces the notes sheet rather than adding duplicates. Visual appearance still requires a separate review.

In [47]:
# Cell 47 — Document, save and verify the workbook

# Calculate warning counts from the current exception table.
warning_events = exceptions_df.loc[
    exceptions_df["severity"].eq("WARNING")
]

warning_event_count = len(warning_events)

warning_record_count = (
    warning_events[
        ["source_file_sha256", "source_record_number"]
    ]
    .drop_duplicates()
    .shape[0]
)

# This manual investigation applies only to the exact source file reviewed.
investigated_source_sha256 = (
    "63c28e8e9d82b79472a7da7a80b3332956b5a6fbb33d6f120f59b29bcb6f0e70"
)

if source_metadata["sha256"] == investigated_source_sha256:
    investigation_note = (
        "For this source fingerprint only, record 1450 reports "
        "loan_to_value_ratio = 727.68. This is a manual review "
        "candidate, not a confirmed error. Ratio-magnitude checks "
        "are not implemented, so this finding is outside the "
        "automated warning count."
    )
else:
    investigation_note = (
        "The previously documented record-1450 finding belongs "
        "to a different source fingerprint and does not apply "
        "automatically to this dataset. Review this source separately."
    )

# Replace the existing notes sheet when this cell is rerun.
if "Source and Notes" in workbook.sheetnames:
    workbook.remove(workbook["Source and Notes"])

notes = workbook.create_sheet("Source and Notes")

documentation = [
    ("Item", "Details"),

    (
        "Project",
        "Mortgage Data Quality & Reporting Pipeline"
    ),

    (
        "Source file",
        source_metadata["filename"]
    ),

    (
        "Source SHA-256",
        source_metadata["sha256"]
    ),

    (
        "Reporting year",
        "2024"
    ),

    (
        "Geography",
        "Miami-Dade County, Florida; county code 12086"
    ),

    (
        "Release",
        "2024 One Year dataset, as shown by the download interface"
    ),

    (
        "Dataset tool",
        "https://ffiec.cfpb.gov/data-browser/data/"
    ),

    (
        "Public data dictionary",
        "https://ffiec.cfpb.gov/documentation/publications/"
        "loan-level-datasets/lar-data-fields/"
    ),

    (
        "Provenance",
        "Public HMDA market data; not Pegasus internal records."
    ),

    (
        "Amounts",
        "Reported loan amounts in USD. Public values are "
        "privacy-modified; totals are not exact transaction amounts."
    ),

    (
        "Originations",
        "Defined as action_taken = 1. "
        "Purchased loans are reported separately."
    ),

    (
        "Focused lender report",
        "Single Family (1-4 Units):Site-Built only. "
        "Not restricted to owner-occupied properties or home purchases."
    ),

    (
        "Validation",
        "PASS means no implemented rule was triggered. "
        "This is not a complete HMDA compliance or underwriting validator."
    ),

    (
        "Warnings",
        f"{warning_event_count:,} warning events affect "
        f"{warning_record_count:,} distinct records. "
        "Records with warnings and no ERROR findings remain "
        "eligible for reporting."
    ),

    (
        "Repeated rows",
        "Exact public-row matches were retained; they do not "
        "prove duplicate borrowers or applications."
    ),

    (
        "Known review candidate",
        investigation_note
    ),

    (
        "Missing-value handling",
        "Blank, NA, and Exempt representations are preserved separately. "
        "Conditional field applicability is not fully validated."
    ),

    (
        "Refresh",
        "Rerun the notebook against the selected input. "
        "The SQLite database stores one replaceable snapshot. "
        "Year, geography and release descriptions are configured "
        "for this project and must be reviewed if its scope changes."
    ),
]

# Populate the notes sheet.
for row in documentation:
    notes.append(row)

notes.column_dimensions["A"].width = 28
notes.column_dimensions["B"].width = 110
notes.freeze_panes = "B2"
notes.sheet_view.showGridLines = False

# Style the header.
for cell in notes[1]:
    cell.fill = PatternFill(
        fill_type="solid",
        fgColor=NAVY
    )
    cell.font = Font(
        bold=True,
        color=WHITE
    )

notes.row_dimensions[1].height = 24

# Format the documentation rows.
for row in notes.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(
            wrap_text=True,
            vertical="top"
        )

        # Keep documentation strings as text.
        if isinstance(cell.value, str):
            cell.data_type = "s"

    notes.row_dimensions[row[0].row].height = 48

    if row[0].value in {"Known review candidate", "Refresh"}:
        notes.row_dimensions[row[0].row].height = 72

# Save the workbook.
workbook.save(EXCEL_PATH)

# Reopen the saved file and verify its contents.
check_workbook = load_workbook(
    EXCEL_PATH,
    read_only=True
)

try:
    expected_sheet_names = [
        sheet_name
        for sheet_name, _, _ in sheet_definitions
    ] + ["Source and Notes"]

    assert check_workbook.sheetnames == expected_sheet_names, (
        "Saved sheet names or order do not match the expected workbook."
    )

    for sheet_name, frame, _ in sheet_definitions:
        sheet = check_workbook[sheet_name]

        saved_header = next(
            sheet.iter_rows(
                min_row=4,
                max_row=4,
                max_col=len(frame.columns),
                values_only=True
            )
        )

        assert list(saved_header) == list(frame.columns), (
            f"Header mismatch in sheet: {sheet_name}"
        )

        assert sheet.max_row == len(frame) + 4, (
            f"Row-count mismatch in sheet: {sheet_name}"
        )

        if frame.empty:
            continue

        for excel_row_number, (expected_row, saved_row) in enumerate(
            zip(
                frame.itertuples(index=False, name=None),
                sheet.iter_rows(
                    min_row=5,
                    max_row=4 + len(frame),
                    max_col=len(frame.columns),
                    values_only=True
                )
            ),
            start=5
        ):
            for column_name, expected, saved in zip(
                frame.columns,
                expected_row,
                saved_row
            ):
                location = (
                    f"{sheet_name}, Excel row {excel_row_number}, "
                    f"column {column_name}"
                )

                if pd.isna(expected) or expected == "":
                    assert saved is None, (
                        f"Expected an empty cell at {location}"
                    )

                elif isinstance(expected, (int, float, np.number)):
                    assert saved is not None, (
                        f"Missing numeric value at {location}"
                    )

                    assert np.isclose(
                        float(saved),
                        float(expected),
                        rtol=1e-12,
                        atol=1e-8
                    ), f"Numeric mismatch at {location}"

                else:
                    assert saved == expected, (
                        f"Value mismatch at {location}"
                    )

    # Verify that the documentation was also saved correctly.
    saved_documentation = list(
        check_workbook["Source and Notes"].iter_rows(
            min_row=1,
            max_row=len(documentation),
            max_col=2,
            values_only=True
        )
    )

    assert saved_documentation == documentation, (
        "Saved documentation does not match the generated notes."
    )

finally:
    check_workbook.close()

print("PASS: Saved sheet names and order verified.")
print("PASS: Saved table headers, row counts, and values verified.")
print("PASS: Source notes and calculated warning counts verified.")
print(f"Warning events: {warning_event_count:,}")
print(f"Distinct records with warnings: {warning_record_count:,}")
print("Workbook:", EXCEL_PATH)
print(f"Size: {EXCEL_PATH.stat().st_size / 1024:.1f} KB")

PASS: Saved sheet names and order verified.
PASS: Saved table headers, row counts, and values verified.
PASS: Source notes and calculated warning counts verified.
Warning events: 1,646
Distinct records with warnings: 1,544
Workbook: /content/mortgage_project/reports/Mortgage_Data_Quality_Report_2024.xlsx
Size: 121.0 KB


### Cell 48 — Verify and download the report backup

**Purpose and method:** Confirm that the workbook, database and CSV reports exist. Copy the original dataset into the project folder and verify its SHA-256 fingerprint. Create a ZIP backup, check its integrity and required contents, then download the workbook and ZIP.

**Expected result / interpretation:** A verified backup containing the source CSV, SQLite database, Excel workbook and eight CSV reports. Download the notebook separately because this cell does not automatically include it. The ZIP is a working backup, not the final GitHub repository package.

In [48]:
import hashlib
import shutil
import zipfile
from pathlib import Path
from google.colab import files

PROJECT_DIRECTORY = Path("/content/mortgage_project")
backup_path = Path("/content/mortgage_project_data_backup.zip")

# Confirm the expected deliverables exist before building the backup.
expected_report_names = [
    "overview",
    "outcomes",
    "loan_purposes",
    "property_segments",
    "lenders_all_originations",
    "lenders_1_to_4_unit_site_built",
    "validation_rule_results",
    "validation_exception_details",
]

required_files = [
    Path(DATA_PATH),
    Path(DB_PATH),
    Path(EXCEL_PATH),
]

required_files.extend(
    Path(REPORT_DIRECTORY) / f"{name}.csv"
    for name in expected_report_names
)

missing_or_empty_files = [
    str(path)
    for path in required_files
    if not path.is_file() or path.stat().st_size == 0
]

if missing_or_empty_files:
    raise FileNotFoundError(
        "Required files are missing or empty. "
        "Run the earlier export cells first:\n"
        + "\n".join(missing_or_empty_files)
    )

print("PASS: Required source and output files exist.")


# Calculate a file's SHA-256 fingerprint.
def calculate_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as source_file:
        for chunk in iter(
            lambda: source_file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# Check that the current source still matches the processed input.
if calculate_sha256(DATA_PATH) != source_metadata["sha256"]:
    raise ValueError(
        "The source CSV has changed since the pipeline ran. "
        "Rerun the pipeline before creating a backup."
    )

# Copy the unchanged source into the project folder.
raw_directory = PROJECT_DIRECTORY / "data" / "raw"
raw_directory.mkdir(parents=True, exist_ok=True)

raw_backup_path = raw_directory / Path(DATA_PATH).name

# Avoid copying a file onto itself.
if Path(DATA_PATH).resolve() != raw_backup_path.resolve():
    shutil.copy2(DATA_PATH, raw_backup_path)

if calculate_sha256(raw_backup_path) != source_metadata["sha256"]:
    raise ValueError(
        "The copied source file does not match the original fingerprint."
    )

print("PASS: Source CSV and backup copy match the recorded fingerprint.")


# Create a fresh archive outside the project directory.
# Rerunning replaces the previous ZIP.
with zipfile.ZipFile(
    backup_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for file_path in sorted(PROJECT_DIRECTORY.rglob("*")):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=file_path.relative_to(
                    PROJECT_DIRECTORY
                ).as_posix()
            )


# Identify the files that must be present inside the ZIP.
required_project_files = [
    raw_backup_path,
    Path(DB_PATH),
    Path(EXCEL_PATH),
]

required_project_files.extend(
    Path(REPORT_DIRECTORY) / f"{name}.csv"
    for name in expected_report_names
)

required_archive_members = {
    path.relative_to(PROJECT_DIRECTORY).as_posix()
    for path in required_project_files
}


# Verify archive integrity and required contents.
with zipfile.ZipFile(backup_path, mode="r") as archive:
    corrupted_member = archive.testzip()

    if corrupted_member is not None:
        raise ValueError(
            f"ZIP integrity check failed for: {corrupted_member}"
        )

    archive_members = set(archive.namelist())

    missing_members = (
        required_archive_members - archive_members
    )

    if missing_members:
        raise ValueError(
            "Required files are missing from the ZIP:\n"
            + "\n".join(sorted(missing_members))
        )

    # Verify the source bytes stored inside the archive.
    source_member = raw_backup_path.relative_to(
        PROJECT_DIRECTORY
    ).as_posix()

    archived_source_digest = hashlib.sha256()

    with archive.open(source_member) as archived_source:
        for chunk in iter(
            lambda: archived_source.read(1024 * 1024),
            b""
        ):
            archived_source_digest.update(chunk)

    if archived_source_digest.hexdigest() != source_metadata["sha256"]:
        raise ValueError(
            "The source CSV inside the ZIP has an unexpected fingerprint."
        )

    archived_file_count = sum(
        not item.is_dir()
        for item in archive.infolist()
    )

print("PASS: ZIP integrity and required contents verified.")
print("PASS: Archived source CSV fingerprint verified.")
print(f"Files in backup: {archived_file_count:,}")
print(f"Workbook: {Path(EXCEL_PATH).name}")
print(f"Backup: {backup_path.name}")
print(
    f"Backup size: "
    f"{backup_path.stat().st_size / (1024 ** 2):.2f} MiB"
)

print(
    "\nSave the updated notebook separately using:"
    "\nFile → Download → Download .ipynb"
)

# Trigger the downloads after successful verification.
files.download(str(EXCEL_PATH))
files.download(str(backup_path))

PASS: Required source and output files exist.
PASS: Source CSV and backup copy match the recorded fingerprint.
PASS: ZIP integrity and required contents verified.
PASS: Archived source CSV fingerprint verified.
Files in backup: 11
Workbook: Mortgage_Data_Quality_Report_2024.xlsx
Backup: mortgage_project_data_backup.zip
Backup size: 11.98 MiB

Save the updated notebook separately using:
File → Download → Download .ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Key findings and interpretation

### Mortgage activity
- There are **33,869 originations**, or **40.96% of all 82,691 reported records**. This is an outcome share, not an approval rate among credit decisions.
- **20,868 records** are denied applications, **11,709** are withdrawals and **5,663** were closed for incompleteness. Do not attribute those outcomes to data-quality defects without supporting evidence.
- Home purchases account for **19,558 originations**, the largest purpose category.
- **610 lenders** have originations; the focused site-built 1-4-unit subset contains **599 lenders**.

### Property mix changes the interpretation
| Category | Originations | Sum of reported amounts (USD) |
|---|---:|---:|
| Site-built, 1-4 units | 33,689 | 15,904,045,000 |
| Multifamily, site-built | 128 | 2,059,750,000 |
| Manufactured, 1-4 units | 52 | 10,220,000 |
| Total | 33,869 | 17,974,015,000 |

Multifamily records represent approximately **0.38% of originations but 11.46% of reported amount**. Separate property categories when comparing lender sizes and average loan amounts. Three loans for one investigated lender total USD 363,525,000 and are all labelled multifamily. That context explains why large amounts deserve investigation rather than automatic deletion; it does not independently verify their correctness.

### Data quality and traceability
- **1,646 warning events** affect **1,544 distinct records**. Counts overlap and cannot be added to obtain unique records.
- Warning categories: **1,124 zero-interest records**, **415 members of exact-row-match groups**, **101 negative-income records**, and **6 unavailable state values**.
- All input records remain accounted for. No source observations were removed or silently imputed.
- Public-row matches were retained because public disclosures do not provide a unique borrower/application key.
- Isolated tests and integration checks demonstrate the implemented rules and repeatable snapshot loading. They do not establish regulatory compliance or production readiness.

## Limitations and known improvements
1. **Public reporting data:** This is US HMDA data, not a Canadian underwriting dataset, Pegasus operational data, or a live CRM feed.
2. **Privacy modifications:** Rounded/grouped values and unavailable identifiers limit deduplication, ratio reconstruction and exact volume estimates. A public row is not a unique borrower.
3. **Selected rules only:** Numeric syntax, geography scope, selected codes and review conditions are checked. Conditional reporting requirements, most categorical codes and numeric plausibility are not comprehensively validated. LEI format is checked without checksum or registry validation.
4. **Missing data:** Blank, NA and Exempt are tracked distinctly; their appropriateness for each record is not fully validated.
5. **Known ratio candidate:** Record 1450 in this specific source fingerprint reports `loan_to_value_ratio = 727.68`. It is documented but not included in the automated warning count. Do not substitute loan amount/property value for the reported ratio: definitions and disclosure transformations may differ.
6. **DTI representation:** Only a subset has exact numeric DTI. The mean of that subset cannot be presented as the mean DTI of the entire population.
7. **Snapshot architecture:** Reruns replace the dedicated database snapshot. There is no historical version store, concurrent ingestion, production scheduler, deployment or operational monitoring. Record numbering and routing assume one source file per run.
8. **Portability and schema changes:** Paths and downloads assume Colab. Later reports depend on fields beyond the initial 15-column check. Changing schemas may require explicit database migrations. Dependencies must be documented for a reproducible installation.
9. **Documentation maintenance:** Chart axis titles are corrected, warning counts are generated dynamically, and the manual investigation is restricted to the matching source fingerprint. Review year, geography and release descriptions when changing the dataset. Do not disable assertions by running Python in optimized mode.
10. **Reporting freshness:** The XLSX and CSV files are exported snapshots. They do not automatically refresh when the database changes. Amounts shown are reported public values, not audited transaction totals.

## What this project demonstrates
Source-aware ingestion, transparent missing-value handling, traceable exception events, tested validation, SQL constraints and joins, transactional loading, reconciliation, property segmentation and formatted reporting.

A defensible description is: **Built and tested a Python/SQL mortgage data-quality and reporting workflow for 82,691 public HMDA records, implementing 16 rules, recording 1,646 review events across 1,544 records, and generating reconciled SQL and Excel reports.**

## Completion checklist
- [x] Profile raw data and preserve original values.
- [x] Implement and test selected validation rules.
- [x] Verify repeatable SQL snapshot loading and reconciliation.
- [x] Export eight report tables and a nine-sheet workbook.
- [x] Add explanations for all 48 executable cells.

  